<a href="https://colab.research.google.com/github/sudedoruk/ESO_DETR/blob/main/ESO_DETR_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESO-DETR: Enhanced Small Object Detection Transformer
**Based on:** "ESO-DETR: An Improved Real-Time Detection Transformer Model
for Enhanced Small Object Detection in UAV Imagery" (Liu et al., Drones 2025)

## Architecture Summary
| Component | Description |
|-----------|-------------|
| **GSHA Block** | Gated Single-Head Attention replaces BasicBlock in ResNet-18 P4/P5 |
| **MMSA** | Multiscale Multihead Self-Attention replaces AIFI in encoder |
| **ESO-FPN** | SPD Conv + LKDA-Fusion fuses P2 info without heavy parameter cost |
| **ESVF Loss** | EMA + Slide weighting on VariFocal Loss for hard samples |

## Paper Results (VisDrone2019)
- mAP@0.5: **41.0%** (+3.9% over RT-DETR-R18 baseline)
- Parameters: **14.9M** (−25% vs baseline)
- FPS: **120** (real-time capable)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Create folder structure
!mkdir -p /content/visdrone/images/train
!mkdir -p /content/visdrone/images/val
!mkdir -p /content/visdrone/annotations/train
!mkdir -p /content/visdrone/annotations/val

# Unzip directly from Drive (this may take a few minutes)
!unzip -q "/content/drive/MyDrive/Colab Notebooks/VisDrone2019-DET-train.zip" -d /content/visdrone_raw/
!unzip -q "/content/drive/MyDrive/Colab Notebooks/VisDrone2019-DET-val.zip" -d /content/visdrone_raw/

# Train images and annotations
!mv /content/visdrone_raw/VisDrone2019-DET-train/images/* /content/visdrone/images/train/
!mv /content/visdrone_raw/VisDrone2019-DET-train/annotations/* /content/visdrone/annotations/train/

# Val images and annotations
!mv /content/visdrone_raw/VisDrone2019-DET-val/images/* /content/visdrone/images/val/
!mv /content/visdrone_raw/VisDrone2019-DET-val/annotations/* /content/visdrone/annotations/val/

!echo "Train images:" && ls /content/visdrone/images/train/ | wc -l
!echo "Train annotations:" && ls /content/visdrone/annotations/train/ | wc -l
!echo "Val images:" && ls /content/visdrone/images/val/ | wc -l
!echo "Val annotations:" && ls /content/visdrone/annotations/val/ | wc -l

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train images:
6471
Train annotations:
6471
Val images:
548
Val annotations:
548


"""## Cell 1 — Install Dependencies"""

In [ ]:
# Run this cell first in Colab
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install pycocotools matplotlib tqdm thop -q
!nvidia-smi  # Verify GPU

Fri Apr 24 08:06:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

"""## Cell 2 — Imports & Global Config"""

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
import numpy as np
import math
import os
import random
import warnings
from collections import defaultdict
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")

# VisDrone class names (index 0 = ignored, 1–10 used for detection)
VISDRONE_CLASSES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor",
]
NUM_CLASSES = 10

# Paper reference metrics for comparison during interpretation
PAPER_REF = {
    "mAP50": 41.0, "mAP50_95": 24.0,
    "precision": 58.2, "recall": 42.2, "params_M": 14.9,
    "per_class": {
        "pedestrian": 42.4, "people": 31.1, "bicycle": 17.0, "car": 78.8,
        "van": 40.1, "truck": 47.8, "tricycle": 26.5,
        "awning-tricycle": 21.7, "bus": 59.8, "motor": 44.9,
    },
}

PyTorch: 2.10.0+cu128 | Device: cuda


"""## Cell 3 — Utility: ConvBNAct"""

In [ ]:
class ConvBNAct(nn.Module):
    """Conv → BatchNorm → Activation (SiLU by default)"""
    def __init__(self, in_ch, out_ch, k=1, s=1, p=0, g=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.act  = nn.SiLU() if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

"""## Cell 4 — CGLU: Convolutional Gated Linear Unit (used in GSHA Block)"""

In [ ]:
class CGLU(nn.Module):
    """
    Convolutional Gated Linear Unit — the 'gate' in the GSHA Block.
    Applies a learnable gating over a depthwise 3×3 conv path.
    (Figure 2 in paper, bottom path after SHA)
    """
    def __init__(self, dim):
        super().__init__()
        self.norm   = nn.LayerNorm(dim)
        self.fc_in  = nn.Linear(dim, dim * 2)
        self.dw     = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)  # DW Conv
        self.fc_out = nn.Linear(dim, dim)
        self.act    = nn.SiLU()

    def forward(self, x):
        B, C, H, W = x.shape
        # LayerNorm in channel-last format
        x_n = self.norm(x.permute(0, 2, 3, 1))          # B,H,W,C
        gate_inp = self.fc_in(x_n)                        # B,H,W,2C
        x1, g    = gate_inp.chunk(2, dim=-1)              # each B,H,W,C

        # Gate via depthwise conv
        g = self.dw(g.permute(0, 3, 1, 2))               # B,C,H,W
        g = g.permute(0, 2, 3, 1)                         # B,H,W,C

        out = self.fc_out(x1 * self.act(g))               # B,H,W,C
        return out.permute(0, 3, 1, 2)                     # B,C,H,W

"""## Cell 5 — GSHA Block (replaces BasicBlock at P4, P5 in backbone)"""

In [ ]:
class GHSABlock(nn.Module):
    """
    Gated Single-Head Attention Block — core backbone unit.

    Equations from paper (Section 2.2):
        Xatt, Xr  = Split(X, [Catt, C−Catt])           (1)
        Attn(Q,K,V) = Softmax(QK^T / sqrt(d)) V        (2)
        X̃att       = Attention(Xatt·WQ, Xatt·WK, Xatt·WV) (3)
        SHA(X)    = Concat(X̃att, Xr) · WO             (4)

    Then adds CGLU and a shortcut.
    Catt = 64 (as stated in paper).
    """
    def __init__(self, in_ch, out_ch, stride=1, catt=64):
        super().__init__()
        self.catt = min(catt, in_ch)

        # Pre-norm + DW conv (BatchNorm → 3×3 DW)
        self.bn1    = nn.BatchNorm2d(in_ch)
        self.dw     = nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False)
        self.bn2    = nn.BatchNorm2d(in_ch)

        # Single-head attention on Catt channels
        self.ln   = nn.LayerNorm(self.catt)
        self.Wq   = nn.Linear(self.catt, self.catt, bias=False)
        self.Wk   = nn.Linear(self.catt, self.catt, bias=False)
        self.Wv   = nn.Linear(self.catt, self.catt, bias=False)
        self.Wo   = nn.Linear(in_ch, out_ch, bias=False)

        # CGLU after SHA
        self.cglu = CGLU(out_ch)

        # Shortcut
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.bn2(self.dw(self.bn1(x)))  # BN→DW→BN
        B, C, H, W = out.shape

        # Split channels  →  Eq (1)
        xatt = out[:, :self.catt].flatten(2).transpose(1, 2)  # B,HW,Catt
        xr   = out[:, self.catt:]                              # B,C-Catt,H,W

        # Single-head self-attention  →  Eq (2)(3)
        xn = self.ln(xatt)
        Q  = self.Wq(xn)
        K  = self.Wk(xn)
        V  = self.Wv(xn)
        scale  = math.sqrt(self.catt)
        attn   = F.softmax(torch.bmm(Q, K.transpose(1, 2)) / scale, dim=-1)
        xatt_o = torch.bmm(attn, V).transpose(1, 2).reshape(B, self.catt, H, W)

        # Concat + WO projection  →  Eq (4)
        concat = torch.cat([xatt_o, xr], dim=1)               # B,C,H,W
        out    = self.Wo(concat.flatten(2).transpose(1, 2))    # B,HW,out_ch
        out    = out.transpose(1, 2).reshape(B, -1, H, W)

        # CGLU + shortcut
        out = out + self.cglu(out) + identity
        return out

"""## Cell 6 — GSHA-ResNet-18 Backbone"""

In [ ]:
class BasicBlock(nn.Module):
    """Standard ResNet-18 BasicBlock (used in Stage 1 & 2)."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.b1 = nn.BatchNorm2d(out_ch)
        self.c2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
            nn.BatchNorm2d(out_ch),
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()

    def forward(self, x):
        out = self.act(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        return self.act(out + self.skip(x))


class GHSAResNet18(nn.Module):
    """
    ResNet-18 with GSHA blocks replacing BasicBlock at Stage 3 (P4) and Stage 4 (P5).
    Returns C2, C3, C4, C5 at strides 4, 8, 16, 32 respectively.
    (Figure 3 in paper)
    """
    def __init__(self):
        super().__init__()
        # Stem: 3×CBN + MaxPool → stride 4
        self.stem = nn.Sequential(
            ConvBNAct(3,  64, 3, 2, 1),
            ConvBNAct(64, 64, 3, 1, 1),
            ConvBNAct(64, 64, 3, 1, 1),
            nn.MaxPool2d(3, 2, 1),
        )
        # Stage 1 — 2×BasicBlock, 64ch (stride 4)
        self.s1 = nn.Sequential(BasicBlock(64, 64),   BasicBlock(64, 64))
        # Stage 2 — 2×BasicBlock, 128ch (stride 8)
        self.s2 = nn.Sequential(BasicBlock(64, 128, stride=2), BasicBlock(128, 128))
        # Stage 3 — 2×GSHA, 256ch (stride 16)  ← P4
        self.s3 = nn.Sequential(GHSABlock(128, 256, stride=2), GHSABlock(256, 256))
        # Stage 4 — 2×GSHA, 512ch (stride 32)  ← P5
        self.s4 = nn.Sequential(GHSABlock(256, 512, stride=2), GHSABlock(512, 512))
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x  = self.stem(x)   # /4
        c2 = self.s1(x)     # B, 64,  H/4,  W/4
        c3 = self.s2(c2)    # B, 128, H/8,  W/8
        c4 = self.s3(c3)    # B, 256, H/16, W/16
        c5 = self.s4(c4)    # B, 512, H/32, W/32
        return c2, c3, c4, c5

"""## Cell 7 — MMSA: Multiscale Multihead Self-Attention (replaces AIFI in encoder)"""

In [ ]:
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style channel attention (SE block)."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Conv2d(ch, max(ch // r, 8), 1), nn.ReLU6(),
            nn.Conv2d(max(ch // r, 8), ch, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(self.gap(x))


class MultiscaleExtractor(nn.Module):
    """
    8-branch dilated depthwise separable convolution for multiscale feature capture.
    Dilation rates 1–8 as described in Section 2.3.
    """
    def __init__(self, ch):
        super().__init__()
        inner = ch // 8
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(ch, inner, 1, bias=False),
                nn.Conv2d(inner, inner, 3, 1, d, dilation=d, groups=inner, bias=False),
                nn.BatchNorm2d(inner), nn.SiLU(),
            ) for d in range(1, 9)
        ])
        self.fuse = nn.Conv2d(ch, ch, 1, bias=False)

    def forward(self, x):
        return self.fuse(torch.cat([b(x) for b in self.branches], dim=1))


class MMSA(nn.Module):
    """
    Multiscale Multihead Self-Attention  (Section 2.3, Figure 4).

    Equations:
        Q       = MFlatten(P5)                   (5)
        K, V    = MFlatten(Multiscale(P5))       (6)
        Attn    = Reshape(MSAttn(Q, K, V))       (7)
        F5      = Attn + Attn_c                  (8)

    where Attn_c is from the channel-attention branch.
    """
    def __init__(self, ch, num_heads=8):
        super().__init__()
        self.nh       = num_heads
        self.hd       = ch // num_heads
        self.ms       = MultiscaleExtractor(ch)
        self.ca       = ChannelAttention(ch)
        self.q_proj   = nn.Linear(ch, ch)
        self.kv_proj  = nn.Linear(ch, ch * 2)
        self.out_proj = nn.Linear(ch, ch)
        self.norm1    = nn.LayerNorm(ch)
        self.norm2    = nn.LayerNorm(ch)
        self.ffn      = nn.Sequential(
            nn.Linear(ch, ch * 4), nn.GELU(), nn.Linear(ch * 4, ch)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        attn_c = self.ca(x)                                   # Eq(8): channel branch

        ms = self.ms(x)                                        # Multiscale features
        q  = self.norm1(x.flatten(2).transpose(1, 2))         # B,HW,C
        kv = self.norm1(ms.flatten(2).transpose(1, 2))        # B,HW,C

        Q        = self.q_proj(q)
        K, V     = self.kv_proj(kv).chunk(2, dim=-1)

        def split_heads(t):
            return t.reshape(B, -1, self.nh, self.hd).transpose(1, 2)

        Q, K, V  = split_heads(Q), split_heads(K), split_heads(V)
        scale    = math.sqrt(self.hd)
        attn     = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) / scale, dim=-1)
        out      = torch.matmul(attn, V)
        out      = out.transpose(1, 2).reshape(B, H * W, C)
        out      = self.out_proj(out).transpose(1, 2).reshape(B, C, H, W)

        # Eq (8): F5 = Attn + Attn_c
        f5 = out + attn_c

        # FFN (post-attention feed-forward)
        f5_flat = f5.flatten(2).transpose(1, 2)
        f5_flat = f5_flat + self.ffn(self.norm2(f5_flat))
        return f5_flat.transpose(1, 2).reshape(B, C, H, W)

"""## Cell 8 — SPD Conv (Space-to-Depth Convolution)"""

In [ ]:
class SPDConv(nn.Module):
    """
    Space-to-Depth Convolution (Section 2.4, Figure 6).
    Maps spatial pixels to the channel dimension (no striding, no pooling),
    then a non-stride 1×1 conv aligns channel count.
    Prevents information loss when downsampling small objects.
    """
    def __init__(self, in_ch, out_ch, scale=2):
        super().__init__()
        self.s    = scale
        self.conv = ConvBNAct(in_ch * scale * scale, out_ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        s = self.s
        # Rearrange: (B,C,H,W) → (B, C·s², H/s, W/s)
        x = x.reshape(B, C, H // s, s, W // s, s)
        x = x.permute(0, 1, 3, 5, 2, 4).contiguous()
        x = x.reshape(B, C * s * s, H // s, W // s)
        return self.conv(x)

"""## Cell 9 — LKDA: Large Kernel Dual-Domain Attention (Section 2.4, Figure 8)"""

In [ ]:
class SCAM(nn.Module):
    """
    Spatial Channel Attention Module.
    XSCAM = XInput ⊗ W1×1(GAP(XInput))    Eq (9)
    """
    def __init__(self, ch):
        super().__init__()
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        return x * self.conv(self.gap(x))


class FCAM(nn.Module):
    """
    Frequency-domain Channel Attention Module.
    XFCAM = IFFT( FFT(XSCAM) ⊗ W1×1(GAP(XSCAM)) )    Eq (10)
    """
    def __init__(self, ch):
        super().__init__()
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        xf  = torch.fft.rfft2(x, norm="ortho")
        w   = torch.sigmoid(self.conv(self.gap(xf.abs())))
        xf  = xf * w
        return torch.fft.irfft2(xf, s=(x.shape[-2], x.shape[-1]), norm="ortho")


class FSAM(nn.Module):
    """
    Frequency-domain Spatial Attention Module.
    XFSAM = α · IFFT(FFT(W1×1(XFCAM)) ⊗ W1×1(XFCAM)) + β · XFCAM    Eq (11)
    """
    def __init__(self, ch):
        super().__init__()
        self.c1    = nn.Conv2d(ch, ch, 1)
        self.c2    = nn.Conv2d(ch, ch, 1)
        self.alpha = nn.Parameter(torch.ones(1))
        self.beta  = nn.Parameter(torch.ones(1))
    def forward(self, x):
        x1 = torch.fft.rfft2(self.c1(x), norm="ortho")
        x2 = self.c2(x)
        x2f = torch.fft.rfft2(x2, norm="ortho")
        out = torch.fft.irfft2(x1 * x2f, s=(x.shape[-2], x.shape[-1]), norm="ortho")
        return self.alpha * out + self.beta * x


class LKDA(nn.Module):
    """
    Large Kernel Dual-domain Attention module (Figure 8).

    Uses four large-kernel DConv branches (1×1, 1×31, 31×1, 31×31) as
    the strip/large convolution kernel component, followed by the
    three-stage dual-domain attention: SCAM → FCAM → FSAM.
    """
    def __init__(self, ch):
        super().__init__()
        # Large-kernel convolution branches (strip + 2D)
        self.dk1x1  = nn.Conv2d(ch, ch, 1,      padding=0,     groups=ch, bias=False)
        self.dk1x31 = nn.Conv2d(ch, ch, (1, 31), padding=(0, 15), groups=ch, bias=False)
        self.dk31x1 = nn.Conv2d(ch, ch, (31, 1), padding=(15, 0), groups=ch, bias=False)
        self.dk31x31= nn.Conv2d(ch, ch, (31, 31), padding=(15, 15), groups=ch, bias=False)
        self.fuse   = ConvBNAct(ch, ch, 1)

        # Dual-domain attention pipeline
        self.scam   = SCAM(ch)
        self.fcam   = FCAM(ch)
        self.fsam   = FSAM(ch)

    def forward(self, x):
        # Large kernel feature extraction
        out = (self.dk1x1(x) + self.dk1x31(x)
               + self.dk31x1(x) + self.dk31x31(x))
        out = self.fuse(out)

        # Dual-domain attention: SCAM → FCAM → FSAM
        out = self.scam(out)
        out = self.fcam(out)
        out = self.fsam(out)
        return out + x   # Residual

"""## Cell 10 — ESO-FPN (Feature Pyramid with LKDA-Fusion)"""

In [ ]:
class RepBlock(nn.Module):
    """Re-parameterizable conv block used inside fusion modules."""
    def __init__(self, ch, n=1):
        super().__init__()
        self.blocks = nn.Sequential(
            *[nn.Sequential(ConvBNAct(ch, ch, 3, 1, 1), ConvBNAct(ch, ch, 3, 1, 1))
              for _ in range(n)]
        )
    def forward(self, x):
        return self.blocks(x)


class FusionBlock(nn.Module):
    """Standard CSP-style feature fusion (Figure 7a)."""
    def __init__(self, in_ch, out_ch, n=1):
        super().__init__()
        self.c1  = ConvBNAct(in_ch, out_ch, 1)
        self.rep = RepBlock(out_ch, n)
        self.c2  = ConvBNAct(in_ch, out_ch, 1)
    def forward(self, x):
        return self.c1(x) + self.rep(self.c1(x)) + self.c2(x)


class LKDAFusion(nn.Module):
    """
    LKDA-Fusion module (Figure 7b, Section 2.4).
    Two parallel paths:
      - Shallow path: 1×1 conv → LKDA  (preserves fine details)
      - Deep    path: 1×1 conv → RepBlock  (reconstructs semantics)
    Fused by element-wise addition.
    """
    def __init__(self, in_ch, out_ch, n=1):
        super().__init__()
        self.shallow = nn.Sequential(ConvBNAct(in_ch, out_ch, 1), LKDA(out_ch))
        self.deep    = nn.Sequential(ConvBNAct(in_ch, out_ch, 1), RepBlock(out_ch, n))
        self.proj    = ConvBNAct(in_ch, out_ch, 1)

    def forward(self, x):
        return self.shallow(x) + self.deep(x) + self.proj(x)


class ESOFPN(nn.Module):
    """
    ESO-FPN: three-level output (P3/P4/P5) that incorporates the P2 (C2)
    fine-grained features through SPD Conv + LKDA-Fusion,
    without the heavy cost of a full four-layer pyramid.
    (Figure 5c in paper)

    Flow:
        C5 → lat5 → P5_td
        C4 + up(P5_td) → fusion4 → P4_td
        C3 + up(P4_td) + SPD(C2) → lkda_fusion3 → P3

        P3 → down → merge with P4_td → P4_bu
        P4_bu → down → merge with P5_td → P5_bu
    """
    def __init__(self, c2=64, c3=128, c4=256, c5=512, d=256):
        super().__init__()
        # Lateral projections
        self.lat5 = ConvBNAct(c5, d, 1)
        self.lat4 = ConvBNAct(c4, d, 1)
        self.lat3 = ConvBNAct(c3, d, 1)

        # SPD Conv: C2 (stride 4) → (stride 8), same spatial as P3
        self.spd  = SPDConv(c2, d, scale=2)

        # Top-down fusion
        self.fus5      = FusionBlock(d,     d)
        self.fus4_td   = FusionBlock(d * 2, d)
        self.lkda_fus3 = LKDAFusion(d * 3,  d)  # P4_up + C3_lat + SPD(C2)

        # Bottom-up fusion
        self.dn3 = ConvBNAct(d, d, 3, 2, 1)
        self.dn4 = ConvBNAct(d, d, 3, 2, 1)
        self.fus4_bu = FusionBlock(d * 2, d)
        self.fus5_bu = FusionBlock(d * 2, d)

    def forward(self, c2, c3, c4, f5):
        # --- Top-down ---
        p5 = self.fus5(self.lat5(f5))

        p5_up = F.interpolate(p5, size=c4.shape[-2:], mode="bilinear", align_corners=False)
        p4_td = self.fus4_td(torch.cat([p5_up, self.lat4(c4)], 1))

        p4_up  = F.interpolate(p4_td, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        c2_spd = self.spd(c2)
        c2_spd = F.interpolate(c2_spd, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        p3 = self.lkda_fus3(torch.cat([p4_up, self.lat3(c3), c2_spd], 1))

        # --- Bottom-up ---
        p4 = self.fus4_bu(torch.cat([self.dn3(p3), p4_td], 1))
        p5 = self.fus5_bu(torch.cat([self.dn4(p4), p5],   1))

        return p3, p4, p5   # strides: 8, 16, 32

"""## Cell 11 — Detection Head & Full ESO-DETR Model"""

In [ ]:
class DetHead(nn.Module):
    """Decoupled classification + box-regression head for one scale."""
    def __init__(self, d, nc):
        super().__init__()
        self.cls = nn.Sequential(
            ConvBNAct(d, d, 3, 1, 1), ConvBNAct(d, d, 3, 1, 1),
            nn.Conv2d(d, nc, 1),
        )
        self.box = nn.Sequential(
            ConvBNAct(d, d, 3, 1, 1), ConvBNAct(d, d, 3, 1, 1),
            nn.Conv2d(d, 4, 1),       # cx, cy, w, h (normalised)
        )
    def forward(self, x):
        return self.cls(x), self.box(x)


class ESODETR(nn.Module):
    """
    Full ESO-DETR model.

    Pipeline:
        Image (B,3,640,640)
          ↓ GHSAResNet18
        C2, C3, C4, C5
          ↓ MMSA (on C5)
        F5
          ↓ ESOFPN
        P3, P4, P5
          ↓ DetHead × 3
        cls_list, box_list  (one tensor per scale)
    """
    def __init__(self, nc=10, d=256):
        super().__init__()
        self.backbone = GHSAResNet18()
        self.mmsa     = MMSA(512, num_heads=8)
        self.fpn      = ESOFPN(c2=64, c3=128, c4=256, c5=512, d=d)
        self.heads    = nn.ModuleList([DetHead(d, nc) for _ in range(3)])
        self._init_heads()

    def _init_heads(self):
        for head in self.heads:
            nn.init.constant_(head.cls[-1].bias, -math.log((1 - 0.01) / 0.01))

    def forward(self, x):
        c2, c3, c4, c5  = self.backbone(x)
        f5               = self.mmsa(c5)
        p3, p4, p5       = self.fpn(c2, c3, c4, f5)

        cls_list, box_list = [], []
        for feat, head in zip([p3, p4, p5], self.heads):
            cls, box = head(feat)
            cls_list.append(cls)
            box_list.append(box)
        return cls_list, box_list

"""## Cell 12 — ESVF Loss (EMASlideVariFocal Loss)"""

In [ ]:
class ESVFLoss(nn.Module):
    """
    EMASlideVariFocal Loss (Section 2.5).

    Extends VariFocal Loss with:
    1. EMA smoothing of IoU threshold:
           IOU_{t+1} = d·IOU_t + (1−d)·IOU_ema             Eq (15)
    2. Slide weight for hard-sample mining:
           w = 1                     if IOU ≤ IOU_ema − 0.1
           w = exp(1 − IOU_ema)      if IOU_ema−0.1 < IOU < IOU_ema
           w = exp(1 − IOU)          if IOU ≥ IOU_ema        Eq (16)
    3. Loss = Σ VFL · w_slide                                Eq (17)
    """
    def __init__(self, nc=10, alpha=0.75, gamma=2.0, ema_d=0.9):
        super().__init__()
        self.nc      = nc
        self.alpha   = alpha
        self.gamma   = gamma
        self.ema_d   = ema_d
        self.strides = [8, 16, 32]
        self.register_buffer("iou_ema", torch.tensor(0.5))

    @staticmethod
    def _iou(pb, gb):
        """Vectorised IoU — both (N,4) cxcywh normalised."""
        def to_xyxy(b):
            return torch.stack([b[:,0]-b[:,2]/2, b[:,1]-b[:,3]/2,
                                 b[:,0]+b[:,2]/2, b[:,1]+b[:,3]/2], -1)
        pb, gb = to_xyxy(pb), to_xyxy(gb)
        ix1 = torch.max(pb[:,0:1], gb[:,0].unsqueeze(0))
        iy1 = torch.max(pb[:,1:2], gb[:,1].unsqueeze(0))
        ix2 = torch.min(pb[:,2:3], gb[:,2].unsqueeze(0))
        iy2 = torch.min(pb[:,3:4], gb[:,3].unsqueeze(0))
        inter = (ix2-ix1).clamp(0) * (iy2-iy1).clamp(0)
        pa = (pb[:,2]-pb[:,0]) * (pb[:,3]-pb[:,1])
        ga = (gb[:,2]-gb[:,0]) * (gb[:,3]-gb[:,1])
        union = pa.unsqueeze(1) + ga.unsqueeze(0) - inter + 1e-7
        return inter / union   # (N_pred, N_gt)

    def _slide_w(self, iou):
        ema = self.iou_ema.item()
        w   = torch.ones_like(iou)
        w[iou > ema - 0.1] = math.exp(1 - ema)
        w[iou >= ema]      = torch.exp(1 - iou[iou >= ema])
        return w

    def _varifocal(self, pred_l, tgt_s, label):
        """VariFocal loss: Eq (12)–(14)."""
        p    = torch.sigmoid(pred_l)
        bce  = F.binary_cross_entropy_with_logits(pred_l, tgt_s, reduction="none")
        wt   = self.alpha * p.detach()**self.gamma * (1 - label) + tgt_s * label
        return (bce * wt).sum()

    def forward(self, cls_list, box_list, targets):
        dev = cls_list[0].device
        cls_l = torch.tensor(0., device=dev)
        box_l = torch.tensor(0., device=dev)
        B     = cls_list[0].shape[0]
        all_ious = []

        for scale_i, (pc, pb, stride) in enumerate(zip(cls_list, box_list, self.strides)):
            _, nc, H, W = pc.shape
            gy, gx = torch.meshgrid(torch.arange(H, device=dev),
                                     torch.arange(W, device=dev), indexing="ij")
            anc_x = (gx.float() + .5) * stride / 640.
            anc_y = (gy.float() + .5) * stride / 640.

            for b in range(B):
                tgt = targets[b]
                if len(tgt["boxes"]) == 0:
                    continue

                gt_b  = tgt["boxes"].to(dev)
                gt_lb = tgt["labels"].to(dev)
                N_gt  = len(gt_b)

                pc_b  = pc[b].permute(1,2,0).reshape(-1, nc)   # HW,nc
                pb_b  = pb[b].permute(1,2,0).reshape(-1, 4)    # HW,4
                ax    = anc_x.reshape(-1)
                ay    = anc_y.reshape(-1)

                cls_tgt = torch.zeros_like(pc_b)
                box_tgt = torch.zeros_like(pb_b)
                is_pos  = torch.zeros(len(pc_b), dtype=torch.bool, device=dev)
                iou_sc  = torch.zeros(len(pc_b), device=dev)

                for gi in range(N_gt):
                    gb = gt_b[gi]
                    x1, y1 = gb[0]-gb[2]/2, gb[1]-gb[3]/2
                    x2, y2 = gb[0]+gb[2]/2, gb[1]+gb[3]/2
                    mask = (ax>x1)&(ax<x2)&(ay>y1)&(ay<y2)
                    if not mask.any():
                        d = ((ax-gb[0])**2+(ay-gb[1])**2); mask[d.argmin()] = True
                    pidx = mask.nonzero(as_tuple=True)[0]
                    cls_tgt[pidx, gt_lb[gi].long()] = 1.
                    box_tgt[pidx] = gb
                    is_pos[pidx]  = True
                    if len(pidx):
                        iou_sc[pidx] = torch.maximum(
                            iou_sc[pidx],
                            self._iou(pb_b[pidx].detach(), gb.unsqueeze(0)).squeeze(1)
                        )

                if is_pos.any():
                    all_ious.append(iou_sc[is_pos].detach())

                sw = self._slide_w(iou_sc)                    # slide weight

                # Classification loss (VFL × slide_w)
                npos = is_pos.sum().item() or 1
                for c in range(nc):
                    vfl = self._varifocal(pc_b[:,c], cls_tgt[:,c], cls_tgt[:,c])
                    cls_l = cls_l + (vfl * sw).sum() / npos

                # Box regression loss (smooth L1) — positives only
                if is_pos.any():
                    box_l = box_l + F.smooth_l1_loss(pb_b[is_pos], box_tgt[is_pos])

        # Update IOU EMA
        if all_ious:
            mean_iou = torch.cat(all_ious).mean()
            self.iou_ema.mul_(self.ema_d).add_((1-self.ema_d) * mean_iou)

        return cls_l, box_l

"""## Cell 13 — VisDrone Dataset"""

In [ ]:
class VisDroneDataset(Dataset):
    """
    VisDrone2019 Object Detection Dataset.

    Annotation format (one .txt per image):
        x_min, y_min, w, h, score, category, truncation, occlusion

    Class mapping (1-indexed → 0-indexed):
        1=pedestrian … 10=motor  (0=ignored, skipped)

    Download:
        https://github.com/VisDrone/VisDrone-Dataset  (Task 1: Images)

    Expected structure:
        <root>/images/{train,val}/
        <root>/annotations/{train,val}/
    """
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, img_dir, ann_dir, img_size=640, augment=False):
        self.img_dir  = img_dir
        self.ann_dir  = ann_dir
        self.img_size = img_size
        self.augment  = augment
        self.files    = sorted(f for f in os.listdir(img_dir) if f.endswith(".jpg"))

    def __len__(self):
        return len(self.files)

    def _parse_ann(self, path, iw, ih):
        boxes, labels = [], []
        if not os.path.exists(path):
            return torch.zeros(0,4), torch.zeros(0, dtype=torch.long)
        for line in open(path):
            p = line.strip().split(",")
            if len(p) < 6: continue
            x,y,w,h = float(p[0]),float(p[1]),float(p[2]),float(p[3])
            score,cat = int(p[4]),int(p[5])
            if cat==0 or score==0 or w<=0 or h<=0: continue
            cx = max(0,min(1,(x+w/2)/iw)); cy = max(0,min(1,(y+h/2)/ih))
            nw = max(0,min(1,w/iw));        nh = max(0,min(1,h/ih))
            if nw>0 and nh>0:
                boxes.append([cx,cy,nw,nh]); labels.append(cat-1)
        if not boxes:
            return torch.zeros(0,4), torch.zeros(0, dtype=torch.long)
        return torch.tensor(boxes,dtype=torch.float32), torch.tensor(labels,dtype=torch.long)

    def __getitem__(self, idx):
        fname   = self.files[idx]
        img     = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        iw, ih  = img.size
        boxes, labels = self._parse_ann(
            os.path.join(self.ann_dir, fname.replace(".jpg", ".txt")), iw, ih)

        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        # Augmentation
        if self.augment:
            if random.random() > .5:            # Horizontal flip
                img = TF.hflip(img)
                if len(boxes): boxes[:,0] = 1 - boxes[:,0]
            img = TF.adjust_brightness(img, 1+random.uniform(-.3,.3))
            img = TF.adjust_contrast(img,   1+random.uniform(-.3,.3))

        t = TF.to_tensor(img)
        t = TF.normalize(t, self.MEAN, self.STD)
        return t, {"boxes": boxes, "labels": labels}


def collate(batch):
    imgs, tgts = zip(*batch)
    return torch.stack(imgs), list(tgts)

"""## Cell 14 — Training Loop"""

In [ ]:
def train_epoch(model, loader, opt, crit, dev, epoch):
    model.train()
    tot, tcls, tbox = 0., 0., 0.
    bar = tqdm(loader, desc=f"Epoch {epoch:03d}", leave=False)
    for imgs, targets in bar:
        imgs    = imgs.to(dev)
        targets = [{k: v.to(dev) for k,v in t.items()} for t in targets]
        cls_l, box_l, *_ = crit(*model(imgs), targets)
        loss = cls_l + 5. * box_l
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot  += loss.item(); tcls += cls_l.item(); tbox += box_l.item()
        bar.set_postfix(loss=f"{loss.item():.3f}",
                         cls=f"{cls_l.item():.3f}", box=f"{box_l.item():.3f}")
    n = len(loader)
    return tot/n, tcls/n, tbox/n


def run_training(cfg):
    """
    cfg = {
        "train_img": "/content/visdrone/images/train",
        "train_ann": "/content/visdrone/annotations/train",
        "val_img":   "/content/visdrone/images/val",
        "val_ann":   "/content/visdrone/annotations/val",
        "save_dir":  "/content/checkpoints",
        "epochs": 100, "batch": 4, "lr": 1e-4, "nc": 10, "d": 256,
    }
    """
    os.makedirs(cfg["save_dir"], exist_ok=True)

    # --- Datasets ---
    train_ds = VisDroneDataset(cfg["train_img"], cfg["train_ann"], augment=True)
    val_ds   = VisDroneDataset(cfg["val_img"],   cfg["val_ann"],   augment=False)
    train_ld = DataLoader(train_ds, cfg["batch"], shuffle=True,
                          num_workers=2, collate_fn=collate, pin_memory=True)
    val_ld   = DataLoader(val_ds,   cfg["batch"], shuffle=False,
                          num_workers=2, collate_fn=collate, pin_memory=True)
    print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

    # --- Model ---
    model = ESODETR(nc=cfg["nc"], d=cfg["d"]).to(device)
    params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {params/1e6:.2f}M  (paper target: 14.9M)")

    # --- Loss & Optimiser (AdamW as in paper) ---
    crit  = ESVFLoss(nc=cfg["nc"]).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=1e-6)

    history = {"loss":[], "cls_loss":[], "box_loss":[], "mAP50":[], "per_class":[]}
    best_map = 0.

    for ep in range(1, cfg["epochs"]+1):
        loss, cl, bl = train_epoch(model, train_ld, opt, crit, device, ep)
        sched.step()
        history["loss"].append(loss)
        history["cls_loss"].append(cl)
        history["box_loss"].append(bl)

        # Validate every 5 epochs
        if ep % 5 == 0 or ep == cfg["epochs"]:
            mAP, ap_cls = evaluate(model, val_ld, device, cfg["nc"])
            history["mAP50"].append(mAP)
            history["per_class"].append(ap_cls)
            tag = " ★" if mAP > best_map else ""
            print(f"[Ep {ep:03d}] loss={loss:.4f}  mAP@0.5={mAP*100:.2f}%{tag}")
            if mAP > best_map:
                best_map = mAP
                torch.save({"epoch": ep, "state": model.state_dict(),
                            "mAP50": mAP, "cfg": cfg},
                           os.path.join(cfg["save_dir"], "best.pth"))
        if ep % 10 == 0:
            torch.save({"epoch": ep, "state": model.state_dict(),
                        "history": history},
                       os.path.join(cfg["save_dir"], f"ckpt_{ep:03d}.pth"))

    print(f"\nDone. Best mAP@0.5 = {best_map*100:.2f}%")
    return model, history

"""## Cell 15 — Evaluation (mAP@0.5)"""

In [ ]:
def _cxcywh_to_xyxy(b, s=640):
    x1=(b[:,0]-b[:,2]/2)*s; y1=(b[:,1]-b[:,3]/2)*s
    x2=(b[:,0]+b[:,2]/2)*s; y2=(b[:,1]+b[:,3]/2)*s
    return torch.stack([x1,y1,x2,y2],-1)


def decode_preds(cls_l, box_l, conf=0.3, nms_t=0.5, s=640, strides=[8,16,32]):
    B   = cls_l[0].shape[0]
    out = []
    for b in range(B):
        bxs, scs, lbs = [], [], []
        for pc, pb, st in zip(cls_l, box_l, strides):
            _, nc, H, W = pc.shape
            gy,gx = torch.meshgrid(torch.arange(H,device=pc.device),
                                    torch.arange(W,device=pc.device), indexing="ij")
            ax = (gx.float()+.5)*st/s
            ay = (gy.float()+.5)*st/s

            sc, lb = torch.sigmoid(pc[b]).max(0)
            mask   = sc.reshape(-1) > conf
            if not mask.any(): continue

            sc = sc.reshape(-1)[mask]; lb = lb.reshape(-1)[mask]
            bx = pb[b].permute(1,2,0).reshape(-1,4)[mask].clone()
            bx[:,0] += ax.reshape(-1)[mask]; bx[:,1] += ay.reshape(-1)[mask]
            bxs.append(_cxcywh_to_xyxy(bx, s).clamp(0,s))
            scs.append(sc); lbs.append(lb)

        if not bxs:
            out.append({"boxes":torch.zeros(0,4),"scores":torch.zeros(0),
                         "labels":torch.zeros(0,dtype=torch.long)}); continue

        bxs=torch.cat(bxs); scs=torch.cat(scs); lbs=torch.cat(lbs)
        keep = torchvision.ops.nms(bxs, scs, nms_t)
        out.append({"boxes":bxs[keep].cpu(),"scores":scs[keep].cpu(),"labels":lbs[keep].cpu()})
    return out


def _iou_mat(pb, gb):
    if len(pb)==0 or len(gb)==0: return torch.zeros(len(pb),len(gb))
    ix1=torch.max(pb[:,None,0],gb[None,:,0]); iy1=torch.max(pb[:,None,1],gb[None,:,1])
    ix2=torch.min(pb[:,None,2],gb[None,:,2]); iy2=torch.min(pb[:,None,3],gb[None,:,3])
    inter=(ix2-ix1).clamp(0)*(iy2-iy1).clamp(0)
    pa=(pb[:,2]-pb[:,0])*(pb[:,3]-pb[:,1])
    ga=(gb[:,2]-gb[:,0])*(gb[:,3]-gb[:,1])
    return inter/(pa[:,None]+ga[None,:]-inter+1e-7)


def evaluate(model, loader, dev, nc=10, iou_t=0.5, conf=0.25):
    model.eval()
    det   = defaultdict(list)   # class → [(score, tp), ...]
    n_gt  = defaultdict(int)

    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Evaluating", leave=False):
            imgs = imgs.to(dev)
            cl, bl = model(imgs)
            preds  = decode_preds(cl, bl, conf=conf)

            for pr, tgt in zip(preds, targets):
                gt_b = _cxcywh_to_xyxy(tgt["boxes"]) if len(tgt["boxes"]) else tgt["boxes"]
                gt_l = tgt["labels"]
                for l in gt_l: n_gt[l.item()] += 1

                for c in range(nc):
                    pm = pr["labels"]==c
                    pb = pr["boxes"][pm]; ps = pr["scores"][pm]
                    gm = gt_l==c; gb = gt_b[gm] if len(gt_b) else torch.zeros(0,4)
                    if len(pb)==0: continue
                    idx = ps.argsort(descending=True); pb=pb[idx]; ps=ps[idx]
                    used=set()
                    for box, sc in zip(pb,ps):
                        if len(gb)==0: det[c].append((sc.item(),0)); continue
                        iou = _iou_mat(box.unsqueeze(0), gb)[0]
                        bi  = iou.argmax().item()
                        tp  = int(iou[bi]>=iou_t and bi not in used)
                        if tp: used.add(bi)
                        det[c].append((sc.item(), tp))

    aps = {}
    for c in range(nc):
        if n_gt[c]==0 or len(det[c])==0: aps[c]=0.; continue
        det[c].sort(key=lambda x:-x[0])
        tp_cs = np.cumsum([d[1] for d in det[c]])
        fp_cs = np.cumsum([1-d[1] for d in det[c]])
        prec  = tp_cs/(tp_cs+fp_cs+1e-7)
        rec   = tp_cs/(n_gt[c]+1e-7)
        ap    = sum(prec[rec>=t].max(initial=0)/11 for t in np.linspace(0,1,11))
        aps[c] = ap

    mAP = float(np.mean(list(aps.values()))) if aps else 0.
    return mAP, aps

"""## Cell 16 — Metric Interpretation"""

In [ ]:
def interpret_metrics(mAP50, per_class_ap, history=None, mAP50_95=None):
    """
    Print a clear, structured interpretation of evaluation results
    relative to the paper's reported numbers.
    """
    print("=" * 62)
    print("  ESO-DETR METRIC INTERPRETATION")
    print("=" * 62)

    # ── Overall ──────────────────────────────────────────────────
    print(f"\n📊  Overall mAP@0.5 : {mAP50*100:.2f}%")
    if mAP50_95 is not None:
        print(f"    Overall mAP@.5:.95: {mAP50_95*100:.2f}%  (paper: 24.0%)")

    delta = mAP50*100 - PAPER_REF["mAP50"]
    marker = "✅ Matches paper" if delta>=0 else (
             "⚠️  Within 3 pp"  if delta>=-3 else "❌ Below paper")
    print(f"\n    vs RT-DETR-R18 baseline  (37.1%) : {mAP50*100-37.1:+.2f} pp")
    print(f"    vs ESO-DETR paper result  (41.0%) : {delta:+.2f} pp  → {marker}")

    # ── Per-class AP table ────────────────────────────────────────
    print("\n📋  Per-class AP@0.5")
    print(f"    {'Class':<20}{'Yours':>8}{'Paper':>8}{'Δ':>8}")
    print("    " + "─" * 44)
    for i, cls in enumerate(VISDRONE_CLASSES):
        ap   = per_class_ap.get(i, 0.) * 100
        ref  = PAPER_REF["per_class"].get(cls, 0.)
        diff = ap - ref
        bar  = "▓"*int(ap/5) + "░"*max(0, 20-int(ap/5))
        print(f"    {cls:<20}{ap:>7.1f}%{ref:>7.1f}%{diff:>+7.1f}pp  {bar}")

    # ── Training curve diagnostics ────────────────────────────────
    if history:
        print("\n📉  Training diagnostics")
        losses = history.get("loss", [])
        if losses:
            pct = (losses[0]-losses[-1])/losses[0]*100 if losses[0]>0 else 0
            print(f"    Loss drop   : {losses[0]:.4f} → {losses[-1]:.4f}  ({pct:.1f}%↓)")
            trend = "✅ Converging" if losses[-1]<losses[0]*0.6 else "⚠️  May need more epochs"
            print(f"    Convergence : {trend}")
        if history.get("mAP50"):
            best_ep = (history["mAP50"].index(max(history["mAP50"]))+1)*5
            print(f"    Best mAP50  : {max(history['mAP50'])*100:.2f}% at epoch ~{best_ep}")

    # ── Metric guide ──────────────────────────────────────────────
    print("""
📖  Metric Reference Guide
    ─────────────────────────────────────────────────────
    mAP@0.5       Correct detection if IoU(pred,gt) ≥ 0.5
                  • 37.1% = RT-DETR-R18 baseline
                  • 41.0% = paper ESO-DETR target
                  • >41%  = you matched or beat the paper!

    mAP@0.5:0.95  Average over IoU [0.5,0.55,...,0.95]
                  Penalises poor localisation; harder metric.
                  • Paper target: 24.0%

    Per-class AP  Tells you which object types are difficult:
                  • Car/Bus: easiest (large, distinct)
                  • Bicycle/Awning-tricycle: hardest (small, occluded)
                  • Low recall on 'people' is normal (dense crowds)

    Precision     TP/(TP+FP) — paper: 58.2%
    Recall        TP/(TP+FN) — paper: 42.2%
                  Low recall → model misses objects (raise conf threshold)
                  Low precision → too many false alarms (lower conf threshold)
    ─────────────────────────────────────────────────────
""")

"""## Cell 17 — Plot Training History"""

In [ ]:
def plot_history(history):
    epochs = list(range(1, len(history["loss"])+1))
    val_ep = list(range(5, len(history["loss"])+1, 5))[:len(history["mAP50"])]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("ESO-DETR Training History", fontsize=14, fontweight="bold")

    # Loss
    ax = axes[0]
    ax.plot(epochs, history["loss"],     label="Total",   lw=2)
    ax.plot(epochs, history["cls_loss"], label="Cls",     lw=1.5, ls="--")
    ax.plot(epochs, history["box_loss"], label="Box × 5", lw=1.5, ls=":")
    ax.set(xlabel="Epoch", ylabel="Loss", title="Training Loss")
    ax.legend(); ax.grid(alpha=.3)

    # mAP
    ax = axes[1]
    ax.plot(val_ep, [v*100 for v in history["mAP50"]], "o-", lw=2, color="steelblue")
    ax.axhline(41.0, ls="--", color="red", label="Paper (41.0%)", alpha=.7)
    ax.axhline(37.1, ls=":",  color="grey", label="Baseline (37.1%)", alpha=.7)
    ax.set(xlabel="Epoch", ylabel="mAP@0.5 (%)", title="Validation mAP@0.5")
    ax.legend(); ax.grid(alpha=.3)

    # Per-class AP (last eval)
    ax = axes[2]
    if history["per_class"]:
        last = history["per_class"][-1]
        aps  = [last.get(i,0)*100 for i in range(NUM_CLASSES)]
        refs = [PAPER_REF["per_class"].get(c,0) for c in VISDRONE_CLASSES]
        x    = range(NUM_CLASSES)
        ax.bar(x, aps,  alpha=.7, label="Yours",     color="steelblue")
        ax.bar(x, refs, alpha=.4, label="Paper ref", color="orange", width=.4)
        ax.set_xticks(list(x))
        ax.set_xticklabels(VISDRONE_CLASSES, rotation=45, ha="right", fontsize=8)
        ax.set(ylabel="AP@0.5 (%)", title="Per-Class AP (last eval)")
        ax.legend(); ax.grid(alpha=.3, axis="y")

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: training_history.png")

"""## Cell 18 — Visualise Detections"""

In [ ]:
def visualise(model, dataset, device, n=3, conf=0.3):
    model.eval()
    cmap = plt.cm.tab10(np.linspace(0,1,NUM_CLASSES))
    idxs = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 2, figsize=(14, 5*n))
    if n==1: axes = axes[None]

    with torch.no_grad():
        for row, idx in enumerate(idxs):
            img_t, tgt = dataset[idx]
            preds = decode_preds(*model(img_t.unsqueeze(0).to(device)), conf=conf)[0]
            s     = 640

            # Denormalise
            mean = torch.tensor([.485,.456,.406]).view(3,1,1)
            std  = torch.tensor([.229,.224,.225]).view(3,1,1)
            img_np = (img_t*std+mean).permute(1,2,0).clamp(0,1).numpy()

            for col, (title, boxes, labels, extra) in enumerate([
                ("Ground Truth", tgt["boxes"], tgt["labels"], None),
                ("ESO-DETR Predictions", preds["boxes"], preds["labels"], preds["scores"]),
            ]):
                ax = axes[row][col]
                ax.imshow(img_np); ax.axis("off"); ax.set_title(f"{title}", fontsize=11)
                for i, (box, label) in enumerate(zip(boxes, labels)):
                    label = label.item()
                    x1=(box[0]-box[2]/2)*s if col==0 else box[0].item()
                    y1=(box[1]-box[3]/2)*s if col==0 else box[1].item()
                    w =(box[2]*s)          if col==0 else (box[2]-box[0]).item()
                    h =(box[3]*s)          if col==0 else (box[3]-box[1]).item()
                    rect = patches.Rectangle((x1,y1),w,h,lw=1.5,
                                              edgecolor=cmap[label], facecolor="none")
                    ax.add_patch(rect)
                    txt = VISDRONE_CLASSES[label]
                    if extra is not None: txt += f" {extra[i]:.2f}"
                    ax.text(x1, y1-2, txt, color=cmap[label], fontsize=6.5, fontweight="bold")

    plt.suptitle("ESO-DETR Detection Results", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("detections.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: detections.png")

## Cell 19 — VisDrone Setup Helper
Run the code below to download the dataset or mount from Drive.


In [ ]:
def visdrone_setup_guide():
    print("""
┌─────────────────────────────────────────────────────────┐
│            VISDRONE DATASET SETUP (Colab)               │
├─────────────────────────────────────────────────────────┤
│  Option A — Google Drive (recommended)                  │
│    from google.colab import drive                       │
│    drive.mount('/content/drive')                        │
│    # copy from Drive to /content/visdrone/              │
│                                                         │
│  Option B — Kaggle                                      │
│    !pip install kaggle                                  │
│    # upload your kaggle.json first                      │
│    !kaggle datasets download -d awsaf49/visdrone-data   │
│    !unzip visdrone-data.zip -d /content/visdrone        │
│                                                         │
│  Option C — Direct download (official)                  │
│    https://github.com/VisDrone/VisDrone-Dataset         │
│    Task 1: Object Detection in Images                   │
│                                                         │
│  Expected structure:                                    │
│    /content/visdrone/images/train/*.jpg                 │
│    /content/visdrone/images/val/*.jpg                   │
│    /content/visdrone/annotations/train/*.txt            │
│    /content/visdrone/annotations/val/*.txt              │
└─────────────────────────────────────────────────────────┘
""")

"""## Cell 20 — Quick Architecture Test (no dataset needed)"""

In [ ]:
def quick_test():
    print("=" * 50)
    print("  ESO-DETR — Architecture Quick Test")
    print("=" * 50)

    model = ESODETR(nc=10, d=256).to(device)
    x     = torch.randn(2, 3, 640, 640).to(device)

    with torch.no_grad():
        cls_l, box_l = model(x)

    print("\n✅  Forward pass OK")
    print("\n  Feature map shapes (batch=2):")
    for i,(c,b) in enumerate(zip(cls_l,box_l)):
        st = [8,16,32][i]
        print(f"    P{i+3} (stride {st:2d}): cls={tuple(c.shape)}  box={tuple(b.shape)}")

    total = sum(p.numel() for p in model.parameters())
    print(f"\n  Parameters : {total/1e6:.2f}M  (paper: 14.9M)")

    try:
        from thop import profile
        flops,_ = profile(model, inputs=(x[:1],), verbose=False)
        print(f"  GFLOPs     : {flops/1e9:.1f}G   (paper:  66.0G)")
    except ImportError:
        print("  GFLOPs     : install 'thop' → !pip install thop")

    print("\n  Module parameter breakdown:")
    for name, mod in [("backbone", model.backbone),
                       ("mmsa",     model.mmsa),
                       ("fpn",      model.fpn),
                       ("heads",    model.heads)]:
        n = sum(p.numel() for p in mod.parameters())
        print(f"    {name:<12}: {n/1e6:.3f}M")

    return model

## Cell 21 — RUN EVERYTHING

Steps:
1. Run the quick test first (no dataset needed).
2. Set up VisDrone dataset.
3. Launch training.
4. Evaluate and interpret results.


In [ ]:
import types

def fixed_forward(self, cls_list, box_list, targets):
    dev   = cls_list[0].device
    cls_l = torch.tensor(0., device=dev)
    box_l = torch.tensor(0., device=dev)
    B     = cls_list[0].shape[0]

    for scale_i, (pc, pb, stride) in enumerate(zip(cls_list, box_list, self.strides)):
        _, nc, H, W = pc.shape
        gy, gx = torch.meshgrid(torch.arange(H, device=dev),
                                 torch.arange(W, device=dev), indexing="ij")
        anc_x = (gx.float() + .5) * stride / 640.
        anc_y = (gy.float() + .5) * stride / 640.

        for b in range(B):
            tgt     = targets[b]
            pc_b    = pc[b].permute(1,2,0).reshape(-1, nc)
            pb_b    = pb[b].permute(1,2,0).reshape(-1, 4)
            n_anc   = len(pc_b)
            cls_tgt = torch.zeros_like(pc_b)
            box_tgt = torch.zeros_like(pb_b)
            is_pos  = torch.zeros(n_anc, dtype=torch.bool, device=dev)

            if len(tgt["boxes"]) > 0:
                gt_b = tgt["boxes"].to(dev)
                gt_l = tgt["labels"].to(dev)
                ax   = anc_x.reshape(-1)
                ay   = anc_y.reshape(-1)
                for gi in range(len(gt_b)):
                    gb       = gt_b[gi]
                    x1,y1    = gb[0]-gb[2]/2, gb[1]-gb[3]/2
                    x2,y2    = gb[0]+gb[2]/2, gb[1]+gb[3]/2
                    mask     = (ax>x1)&(ax<x2)&(ay>y1)&(ay<y2)
                    if not mask.any():
                        mask[((ax-gb[0])**2+(ay-gb[1])**2).argmin()] = True
                    pidx     = mask.nonzero(as_tuple=True)[0]
                    cls_tgt[pidx, gt_l[gi].long()] = 1.
                    box_tgt[pidx] = gb
                    is_pos[pidx]  = True

            cls_l = cls_l + F.binary_cross_entropy_with_logits(
                pc_b, cls_tgt, reduction="sum") / n_anc
            if is_pos.any():
                box_l = box_l + F.smooth_l1_loss(
                    pb_b[is_pos], box_tgt[is_pos], reduction="mean")

    return cls_l / (B * len(self.strides)), box_l

ESVFLoss.forward = fixed_forward
print("✅ Loss function patched")

✅ Loss function patched


In [ ]:
torch.cuda.empty_cache()

# Datasets
train_ds = VisDroneDataset(
    "/content/visdrone/images/train",
    "/content/visdrone/annotations/train",
    img_size=416, augment=True
)
val_ds = VisDroneDataset(
    "/content/visdrone/images/val",
    "/content/visdrone/annotations/val",
    img_size=416, augment=False
)

train_ld = DataLoader(train_ds, batch_size=2, shuffle=True,
                      num_workers=2, collate_fn=collate, pin_memory=True)
val_ld   = DataLoader(val_ds,   batch_size=2, shuffle=False,
                      num_workers=2, collate_fn=collate, pin_memory=True)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

# Model
model = ESODETR(nc=10, d=256).to(device)
crit  = ESVFLoss(nc=10).to(device)
ESVFLoss.forward = fixed_forward
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 10, eta_min=1e-6)

os.makedirs("/content/checkpoints", exist_ok=True)
os.makedirs("/content/drive/MyDrive/eso_detr_checkpoints", exist_ok=True)

history = {"loss":[], "cls_loss":[], "box_loss":[], "mAP50":[], "per_class":[]}
ap_cls  = {}
mAP     = 0.

for ep in range(1, 111):
    loss, cl, bl = train_epoch(model, train_ld, opt, crit, device, ep)
    sched.step()
    history["loss"].append(loss)
    history["cls_loss"].append(cl)
    history["box_loss"].append(bl)
    print(f"Epoch {ep:02d} — loss={loss:.4f}")

    # ── Save after EVERY epoch to Drive ──────────────────────
    torch.save({
        "epoch": ep,
        "model": model.state_dict(),
        "opt":   opt.state_dict(),
        "history": history,
    }, f"/content/drive/MyDrive/eso_detr_checkpoints/epoch_{ep:02d}.pth")
    print(f"  💾 Saved epoch {ep} to Drive")

    # ── Validate every 5 epochs ───────────────────────────────
    if ep % 5 == 0:
        mAP, ap_cls = evaluate(model, val_ld, device, nc=10, conf=0.01)
        history["mAP50"].append(mAP)
        history["per_class"].append(ap_cls)
        print(f"[Ep {ep:03d}] mAP@0.5={mAP*100:.2f}%")

# Final results
interpret_metrics(mAP, ap_cls, history)
plot_history(history)
visualise(model, val_ds, device)


Train: 6,471  |  Val: 548


KeyboardInterrupt: 

In [ ]:
#kaldığı yerden devam etmesi için
# Check what epochs are saved
files = sorted([f for f in os.listdir(
    "/content/drive/MyDrive/eso_detr_checkpoints")
    if f.startswith("epoch_")])
print("Saved epochs:", files)

# Load the last one
last_file = files[-1]
ckpt      = torch.load(
    f"/content/drive/MyDrive/eso_detr_checkpoints/{last_file}",
    map_location=device, weights_only=False)

last_epoch = ckpt["epoch"]
history    = ckpt["history"]
print(f"✅ Resuming from epoch {last_epoch}")

# Rebuild model and load weights
train_ds = VisDroneDataset(
    "/content/visdrone/images/train",
    "/content/visdrone/annotations/train",
    img_size=640, augment=True
)
val_ds = VisDroneDataset(
    "/content/visdrone/images/val",
    "/content/visdrone/annotations/val",
    img_size=640, augment=False
)
train_ld = DataLoader(train_ds, batch_size=2, shuffle=True,
                      num_workers=2, collate_fn=collate, pin_memory=True)
val_ld   = DataLoader(val_ds, batch_size=2, shuffle=False,
                      num_workers=2, collate_fn=collate, pin_memory=True)

model = ESODETR(nc=10, d=256).to(device)
model.load_state_dict(ckpt["model"])

crit  = ESVFLoss(nc=10).to(device)
ESVFLoss.forward = fixed_forward
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
opt.load_state_dict(ckpt["opt"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, 50, eta_min=1e-6, last_epoch=last_epoch)

print(f"✅ Model ready — continuing from epoch {last_epoch+1}")

# Continue training
ap_cls   = {}
mAP      = 0.
best_map = 0.

for ep in range(last_epoch+1, 51):
    loss, cl, bl = train_epoch(model, train_ld, opt, crit, device, ep)
    sched.step()
    history["loss"].append(loss)
    history["cls_loss"].append(cl)
    history["box_loss"].append(bl)
    print(f"Epoch {ep:02d} — loss={loss:.4f}")

    torch.save({
        "epoch"  : ep,
        "model"  : model.state_dict(),
        "opt"    : opt.state_dict(),
        "history": history,
    }, f"/content/drive/MyDrive/eso_detr_checkpoints/epoch_{ep:02d}.pth")
    print(f"  💾 Saved epoch {ep}")

    if ep % 5 == 0:
        mAP, ap_cls = evaluate(model, val_ld, device, nc=10, conf=0.01)
        history["mAP50"].append(mAP)
        history["per_class"].append(ap_cls)
        print(f"[Ep {ep:03d}] mAP@0.5={mAP*100:.2f}%")
        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(),
                "/content/drive/MyDrive/eso_detr_checkpoints/best.pth")
            print("  ★ New best saved!")

interpret_metrics(best_map, ap_cls, history)
plot_history(history)
visualise(model, val_ds, device)

Saved epochs: ['epoch_01.pth', 'epoch_02.pth', 'epoch_03.pth', 'epoch_04.pth', 'epoch_05.pth', 'epoch_06.pth', 'epoch_07.pth', 'epoch_08.pth', 'epoch_09.pth', 'epoch_10.pth', 'epoch_11.pth', 'epoch_12.pth', 'epoch_13.pth', 'epoch_14.pth', 'epoch_15.pth']
✅ Resuming from epoch 15
✅ Model ready — continuing from epoch 16


Epoch 16 — loss=0.2978
  💾 Saved epoch 16


Epoch 17 — loss=0.2645
  💾 Saved epoch 17


Epoch 18 — loss=0.2492
  💾 Saved epoch 18


Epoch 19 — loss=0.2375
  💾 Saved epoch 19


...

Epoch 24 — loss=0.1933
  💾 Saved epoch 24


Epoch 25 — loss=0.1864
  💾 Saved epoch 25


[Ep 025] mAP@0.5=0.00%


Epoch 26 — loss=0.1802
  💾 Saved epoch 26


KeyboardInterrupt: 

In [ ]:
model.eval()
imgs, targets = next(iter(val_ld))
imgs = imgs.to(device)

with torch.no_grad():
    cls_l, box_l = model(imgs)

print("=== CLASSIFICATION SCORES ===")
for i, pc in enumerate(cls_l):
    scores = torch.sigmoid(pc[0]).max(0)[0].reshape(-1)
    print(f"P{i+3}: max={scores.max():.4f}  "
          f"mean={scores.mean():.4f}  "
          f"above 0.01: {(scores>0.01).sum().item()}")

print("\n=== BOX PREDICTIONS ===")
for i, pb in enumerate(box_l):
    b = pb[0]
    print(f"P{i+3}: min={b.min():.4f}  "
          f"max={b.max():.4f}  "
          f"mean={b.mean():.4f}")

print("\n=== GROUND TRUTH ===")
print(f"GT boxes: {targets[0]['boxes'][:3]}")
print(f"GT labels: {targets[0]['labels'][:3]}")

=== CLASSIFICATION SCORES ===
P3: max=0.9969  mean=0.0336  above 0.01: 960
P4: max=0.9974  mean=0.0447  above 0.01: 300
P5: max=0.9864  mean=0.0921  above 0.01: 126

=== BOX PREDICTIONS ===
P3: min=-0.0867  max=0.9919  mean=0.2706
P4: min=-0.0401  max=0.9850  mean=0.2664
P5: min=-0.0241  max=0.9952  mean=0.2648

=== GROUND TRUTH ===
GT boxes: tensor([[0.4677, 0.5722, 0.0281, 0.0852],
        [0.5099, 0.5907, 0.0323, 0.0852],
        [0.4727, 0.7037, 0.0349, 0.1019]])
GT labels: tensor([3, 3, 3])


In [ ]:
#decoder düzeltmesi

def decode_preds(cls_l, box_l, conf=0.01, nms_t=0.5, s=640, strides=[8,16,32]):
    B   = cls_l[0].shape[0]
    out = []

    for b in range(B):
        bxs, scs, lbs = [], [], []

        for pc, pb, st in zip(cls_l, box_l, strides):
            _, nc, H, W = pc.shape

            # Build anchor grid
            gy, gx = torch.meshgrid(
                torch.arange(H, device=pc.device),
                torch.arange(W, device=pc.device),
                indexing="ij")

            # Anchor centers in pixel coordinates
            ax = (gx.float() + 0.5) * st  # pixels
            ay = (gy.float() + 0.5) * st  # pixels

            # Get scores and labels
            scores = torch.sigmoid(pc[b])  # nc, H, W
            sc, lb = scores.max(0)         # H, W
            sc_flat = sc.reshape(-1)
            lb_flat = lb.reshape(-1)
            mask    = sc_flat > conf

            if not mask.any():
                continue

            # Get box predictions for confident anchors
            pb_flat = pb[b].permute(1,2,0).reshape(-1, 4)  # HW, 4
            ax_flat = ax.reshape(-1)
            ay_flat = ay.reshape(-1)

            # Filter by confidence
            pb_sel = pb_flat[mask]   # predicted offsets
            ax_sel = ax_flat[mask]   # anchor x centers
            ay_sel = ay_flat[mask]   # anchor y centers
            sc_sel = sc_flat[mask]
            lb_sel = lb_flat[mask]

            # Decode boxes:
            # cx, cy are offsets from anchor center
            # w, h are relative to image size
            cx = ax_sel + pb_sel[:, 0] * st   # absolute cx in pixels
            cy = ay_sel + pb_sel[:, 1] * st   # absolute cy in pixels
            w  = pb_sel[:, 2].clamp(0.01, 1.0) * s   # absolute width
            h  = pb_sel[:, 3].clamp(0.01, 1.0) * s   # absolute height

            # Convert to x1y1x2y2
            x1 = (cx - w/2).clamp(0, s)
            y1 = (cy - h/2).clamp(0, s)
            x2 = (cx + w/2).clamp(0, s)
            y2 = (cy + h/2).clamp(0, s)

            boxes = torch.stack([x1, y1, x2, y2], dim=1)

            # Filter out zero size boxes
            valid = (x2 > x1) & (y2 > y1)
            if not valid.any():
                continue

            bxs.append(boxes[valid])
            scs.append(sc_sel[valid])
            lbs.append(lb_sel[valid])

        if not bxs:
            out.append({
                "boxes":  torch.zeros(0, 4),
                "scores": torch.zeros(0),
                "labels": torch.zeros(0, dtype=torch.long)
            })
            continue

        bxs = torch.cat(bxs)
        scs = torch.cat(scs)
        lbs = torch.cat(lbs)

        # NMS
        keep = torchvision.ops.nms(bxs, scs, nms_t)
        out.append({
            "boxes":  bxs[keep].cpu(),
            "scores": scs[keep].cpu(),
            "labels": lbs[keep].cpu()
        })

    return out

print("✅ decode_preds fixed")

✅ decode_preds fixed


In [ ]:
# Quick evaluation test
mAP, ap_cls = evaluate(model, val_ld, device, nc=10, conf=0.01)
print(f"mAP@0.5 = {mAP*100:.2f}%")
for i, cls in enumerate(VISDRONE_CLASSES):
    ap = ap_cls.get(i, 0) * 100
    print(f"  {cls:<20}: {ap:.2f}%")

mAP@0.5 = 5.80%
  pedestrian          : 4.80%
  people              : 0.92%
  bicycle             : 0.15%
  car                 : 16.64%
  van                 : 4.02%
  truck               : 9.60%
  tricycle            : 9.09%
  awning-tricycle     : 1.21%
  bus                 : 6.54%
  motor               : 5.01%


In [ ]:
#kaldığın yerden devam
# Continue training from epoch 25
ap_cls   = {}
mAP      = 0.
best_map = 0.

for ep in range(26, 51):
    loss, cl, bl = train_epoch(model, train_ld, opt, crit, device, ep)
    sched.step()
    history["loss"].append(loss)
    history["cls_loss"].append(cl)
    history["box_loss"].append(bl)
    print(f"Epoch {ep:02d} — loss={loss:.4f}")

    # Save every epoch
    torch.save({
        "epoch"  : ep,
        "model"  : model.state_dict(),
        "opt"    : opt.state_dict(),
        "history": history,
    }, f"/content/drive/MyDrive/eso_detr_checkpoints/epoch_{ep:02d}.pth")
    print(f"  💾 Saved epoch {ep}")

    # Validate every 5 epochs
    if ep % 5 == 0:
        mAP, ap_cls = evaluate(model, val_ld, device, nc=10, conf=0.01)
        history["mAP50"].append(mAP)
        history["per_class"].append(ap_cls)
        print(f"[Ep {ep:03d}] mAP@0.5={mAP*100:.2f}%")
        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(),
                "/content/drive/MyDrive/eso_detr_checkpoints/best.pth")
            print("  ★ New best saved!")

interpret_metrics(best_map, ap_cls, history)
plot_history(history)
visualise(model, val_ds, device)

Epoch 26 — loss=0.1748
  💾 Saved epoch 26


Epoch 27 — loss=0.1686
  💾 Saved epoch 27


Epoch 28 — loss=0.1635
  💾 Saved epoch 28


Epoch 29 — loss=0.1578
  💾 Saved epoch 29


Epoch 30 — loss=0.1529
  💾 Saved epoch 30


...

Epoch 32 — loss=0.1444
  💾 Saved epoch 32


Epoch 33 — loss=0.1401
  💾 Saved epoch 33


Epoch 34 — loss=0.1363
  💾 Saved epoch 34


Epoch 35 — loss=0.1329
  💾 Saved epoch 35


Evaluating:  99%|█████████▉| 272/274 [01:16<00:00,  4.82it/s]

In [ ]:
# Check what raw boxes look like
model.eval()
imgs, targets = next(iter(val_ld))
imgs = imgs.to(device)

with torch.no_grad():
    cls_l, box_l = model(imgs)

# Check box prediction ranges
for i, (pc, pb) in enumerate(zip(cls_l, box_l)):
    scores = torch.sigmoid(pc[0]).max(0)[0]
    boxes  = pb[0]
    print(f"Scale P{i+3}:")
    print(f"  Max confidence : {scores.max():.4f}")
    print(f"  Box values min : {boxes.min():.4f}")
    print(f"  Box values max : {boxes.max():.4f}")
    print(f"  Box values mean: {boxes.mean():.4f}")

Scale P3:
  Max confidence : 0.0178
  Box values min : -0.1746
  Box values max : 0.5826
  Box values mean: 0.0540
Scale P4:
  Max confidence : 0.0174
  Box values min : -0.2184
  Box values max : 0.4283
  Box values mean: 0.0348
Scale P5:
  Max confidence : 0.0157
  Box values min : -0.2601
  Box values max : 0.3631
  Box values mean: -0.0262


In [ ]:
# Load pretrained ResNet18 backbone weights
import torchvision.models as tvm

pretrained = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
backbone_sd = pretrained.state_dict()

# Map pretrained weights to our backbone
our_sd = model.backbone.state_dict()
matched = 0
for k, v in our_sd.items():
    # Match stem and early stage weights
    if "stem.0" in k:
        pk = "conv1.weight"
        if pk in backbone_sd and backbone_sd[pk].shape == v.shape:
            our_sd[k] = backbone_sd[pk]; matched += 1
    elif "s1.0" in k or "s1.1" in k:
        pk = k.replace("s1.0.c1", "layer1.0.conv1")\
              .replace("s1.0.b1", "layer1.0.bn1")\
              .replace("s1.0.c2", "layer1.0.conv2")\
              .replace("s1.0.b2", "layer1.0.bn2")
        if pk in backbone_sd and backbone_sd[pk].shape == v.shape:
            our_sd[k] = backbone_sd[pk]; matched += 1

model.backbone.load_state_dict(our_sd, strict=False)
print(f"✅ Loaded {matched} pretrained weights into backbone")

# Now retrain from epoch 1 with pretrained backbone
# The model will start much stronger

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 244MB/s]


✅ Loaded 12 pretrained weights into backbone


In [ ]:
# Create the Drive checkpoint folder
import os
os.makedirs("/content/drive/MyDrive/eso_detr_checkpoints", exist_ok=True)
print("✅ Folder created")

# Test with lower confidence to check if model is learning
mAP, ap_cls = evaluate(model, val_ld, device, nc=10, conf=0.05)
print(f"mAP@0.5 (conf=0.05): {mAP*100:.2f}%")

In [ ]:
%%javascript
function KeepAlive() {
  document.querySelector("#top-toolbar").click();
}
setInterval(KeepAlive, 60000);

# **RT-DETR ENTEGRESİYLE**

In [ ]:
#RT-DETR DEN ENTEGRE ETME

# Test if we can download official RT-DETR
!pip install ultralytics -q
from ultralytics import RTDETR

# Load pretrained RT-DETR
model_rt = RTDETR("rtdetr-l.pt")

# Quick validation on VisDrone
results = model_rt.val(
    data="VisDrone.yaml",
    imgsz=640,
    batch=4,
    device=0
)
print(results)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
rt-detr-l summary: 294 layers, 32,148,140 parameters, 0 gradients, 103.8 GFLOPs

WARNING ⚠️ Dataset 'VisDrone.yaml' images not found, missing path '/content/datasets/VisDrone/images/val'
Unzipping /content/datasets/VisDrone/VisDrone2019-DET-val.zip to /content/datasets/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.9Kfiles/s 0.6s
Unzipping /content/datasets/VisDrone/VisDrone2019-DET-test-dev.zip to /content/datasets/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.6Kfiles/s 2.1s
Unzippi

In [ ]:
import yaml
with open("/content/datasets/VisDrone.yaml") as f:
    print(yaml.safe_load(f))

FileNotFoundError: [Errno 2] No such file or directory: '/content/datasets/VisDrone.yaml'

In [ ]:
import ultralytics, os

# Find VisDrone.yaml
ultralytics_path = os.path.dirname(ultralytics.__file__)
for root, dirs, files in os.walk(ultralytics_path):
    for f in files:
        if "VisDrone" in f:
            print(os.path.join(root, f))

/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml


In [ ]:
!ls /content/datasets/

VisDrone


**#RT-DETR EĞİTİM**

In [ ]:


from ultralytics import RTDETR

# Load pretrained RT-DETR
model_rt = RTDETR("rtdetr-l.pt")

# Fine-tune on VisDrone
model_rt.train(
    data="/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml",
    epochs=50,
    imgsz=640,
    batch=8,       # doubled
    device=0,
    lr0=2e-4,      # doubled to match batch size
    optimizer="AdamW",
    workers=4,
    project="/content/drive/MyDrive/rtdetr_visdrone",
    name="rtdetr_fast",
    exist_ok=True,
    save=True,
    save_period=5,
    patience=20,
)

Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_fast, nbs=64, nms=False, opset=None, optimize=F

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      18.8G      1.888     0.2702     0.4426        526        640: 100% ━━━━━━━━━━━━ 809/809 2.0it/s 6:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.8it/s 9.3s
                   all        548      38759      0.325     0.0963     0.0263     0.0116

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      7.45G      1.708      0.403     0.3238        741        640: 0% ──────────── 0/809  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      15.5G      1.467      0.455      0.219        541        640: 100% ━━━━━━━━━━━━ 809/809 2.2it/s 6:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 4.4it/s 8.0s
                   all        548      38759      0.453      0.103     0.0325     0.0142

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      7.29G      1.352     0.4543     0.1522        447        640: 0% ──────────── 0/809  0.4s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      49/50      16.9G     0.7241     0.4504    0.06113        231        640: 100% ━━━━━━━━━━━━ 809/809 2.4it/s 5:37
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 4.3it/s 8.1s
                   all        548      38759       0.57      0.431      0.401      0.233

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      50/50      6.44G      0.725      0.442    0.05078        213        640: 0% ──────────── 0/809  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      50/50      17.2G     0.7229     0.4496    0.06138        357        640: 100% ━━━━━━━━━━━━ 809/809 2.4it/s 5:36
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 4.3it/s 8.1s
                   all        548      38759      0.567      0.429      0.401      0.234

50 epochs completed in 5.125 hours.
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/best.pt...
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
rt-detr-l summary: 310 layers, 32,004,290 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.4it/s 14.5s
                   all        548      38759      0.569  

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ac4710221b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
from ultralytics import RTDETR
import torch

# Load our trained model
model_rt = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/best.pt")

# See the model structure
print(model_rt.model)

RTDETRDetectionModel(
  (model): Sequential(
    (0): HGStem(
      (stem1): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (stem2a): Conv(
        (conv): Conv2d(32, 16, kernel_size=(2, 2), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (stem2b): Conv(
        (conv): Conv2d(16, 32, kernel_size=(2, 2), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (stem3): Conv(
        (conv): Conv2d(64, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
   

In [ ]:
# List all named modules
for name, module in model_rt.model.named_modules():
    print(f"{name}: {type(module).__name__}")

: RTDETRDetectionModel
model: Sequential
model.0: HGStem
model.0.stem1: Conv
model.0.stem1.conv: Conv2d
model.0.stem1.bn: BatchNorm2d
model.0.stem1.act: ReLU
model.0.stem2a: Conv
model.0.stem2a.conv: Conv2d
model.0.stem2a.bn: BatchNorm2d
model.0.stem2a.act: ReLU
model.0.stem2b: Conv
model.0.stem2b.conv: Conv2d
model.0.stem2b.bn: BatchNorm2d
model.0.stem2b.act: ReLU
model.0.stem3: Conv
model.0.stem3.conv: Conv2d
model.0.stem3.bn: BatchNorm2d
model.0.stem3.act: ReLU
model.0.stem4: Conv
model.0.stem4.conv: Conv2d
model.0.stem4.bn: BatchNorm2d
model.0.stem4.act: ReLU
model.0.pool: MaxPool2d
model.1: HGBlock
model.1.m: ModuleList
model.1.m.0: Conv
model.1.m.0.conv: Conv2d
model.1.m.0.bn: BatchNorm2d
model.1.m.0.act: ReLU
model.1.m.1: Conv
model.1.m.1.conv: Conv2d
model.1.m.1.bn: BatchNorm2d
model.1.m.2: Conv
model.1.m.2.conv: Conv2d
model.1.m.2.bn: BatchNorm2d
model.1.m.3: Conv
model.1.m.3.conv: Conv2d
model.1.m.3.bn: BatchNorm2d
model.1.m.4: Conv
model.1.m.4.conv: Conv2d
model.1.m.4.bn: Ba

In [ ]:
from ultralytics import RTDETR
model_rt = RTDETR("rtdetr-r18.pt")
print("✅ Model loaded")

FileNotFoundError: [Errno 2] No such file or directory: 'rtdetr-r18.pt'

In [ ]:
#ESO EKLEME

!pip install ultralytics -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from ultralytics import RTDETR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ── 1. Load trained model ──────────────────────────────────────
model_rt = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/best.pt")
torch_model = model_rt.model.model

# ── 2. Define MMSA to replace AIFI ────────────────────────────
class MMSA_Module(nn.Module):
    """Replaces AIFI (model.11) with Multiscale Multihead Self-Attention"""
    def __init__(self, dim=256, num_heads=8):
        super().__init__()
        self.nh  = num_heads
        self.hd  = dim // num_heads #dim=kanal sayısı

        # Multiscale branch — 4 dilation rates
        self.ms_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(dim, dim//4, 1, bias=False),
                nn.Conv2d(dim//4, dim//4, 3, 1, d,
                          dilation=d, groups=dim//4, bias=False),
                nn.BatchNorm2d(dim//4), nn.SiLU()
            ) for d in [1, 2, 4, 8]
        ])
        self.ms_fuse = nn.Conv2d(dim, dim, 1, bias=False)

        # Channel attention branch
        self.ca_gap  = nn.AdaptiveAvgPool2d(1)
        self.ca_fc   = nn.Sequential(
            nn.Conv2d(dim, dim//16, 1), nn.ReLU(),
            nn.Conv2d(dim//16, dim, 1), nn.Sigmoid()
        )

        # Attention projections
        self.q  = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim*2)
        self.op = nn.Linear(dim, dim)

        # FFN
        self.n1  = nn.LayerNorm(dim)
        self.n2  = nn.LayerNorm(dim)
        self.ff  = nn.Sequential(
            nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))

    def forward(self, x, pos=None):
        B, C, H, W = x.shape

        # Multiscale features
        ms = self.ms_fuse(torch.cat([c(x) for c in self.ms_convs], 1))

        # Channel attention
        ca = x * self.ca_fc(self.ca_gap(x))

        # Flatten for attention
        q_in  = self.n1(x.flatten(2).transpose(1,2))
        kv_in = self.n1(ms.flatten(2).transpose(1,2))

        Q      = self.q(q_in)
        K, V   = self.kv(kv_in).chunk(2, -1)

        def split(t):
            return t.reshape(B,-1,self.nh,self.hd).transpose(1,2)

        Q,K,V  = split(Q), split(K), split(V)
        attn   = F.softmax(Q@K.transpose(-2,-1)/math.sqrt(self.hd), -1)
        out    = (attn@V).transpose(1,2).reshape(B,H*W,C)
        out    = self.op(out).transpose(1,2).reshape(B,C,H,W)

        # F5 = Attn + Channel_attn (Eq.8 in paper)
        f5     = out + ca

        # FFN
        f5f    = f5.flatten(2).transpose(1,2)
        f5f    = f5f + self.ff(self.n2(f5f))
        return f5f.transpose(1,2).reshape(B,C,H,W)

# ── 3. Define GSHA wrapper ─────────────────────────────────────
class GHSAWrapper(nn.Module):
    """Wraps existing HGBlock and adds GSHA attention on top"""
    def __init__(self, hgblock, ch):
        super().__init__()
        self.block = hgblock
        catt       = min(64, ch)

        # Single-head attention on partial channels
        self.ln   = nn.LayerNorm(catt)
        self.Wq   = nn.Linear(catt, catt, bias=False)
        self.Wk   = nn.Linear(catt, catt, bias=False)
        self.Wv   = nn.Linear(catt, catt, bias=False)
        self.Wo   = nn.Linear(ch, ch, bias=False)
        self.catt = catt

        # Gated unit
        self.gate = nn.Sequential(
            nn.LayerNorm(ch),
            nn.Linear(ch, ch*2),
            nn.SiLU()
        )
        self.gate_out = nn.Linear(ch, ch)

    def forward(self, x):
        x = self.block(x)
        B, C, H, W = x.shape

        # Split channels for SHA
        xatt = x[:,:self.catt].flatten(2).transpose(1,2)  # B,HW,catt
        xr   = x[:,self.catt:]                             # B,C-catt,H,W

        xn = self.ln(xatt)
        Q  = self.Wq(xn); K = self.Wk(xn); V = self.Wv(xn)
        a  = F.softmax(Q@K.transpose(-2,-1)/math.sqrt(self.catt), -1)
        xa = (a@V).transpose(1,2).reshape(B,self.catt,H,W)

        # Concat + project
        cat = torch.cat([xa,xr],1)
        out = self.Wo(cat.flatten(2).transpose(1,2))
        out = out.transpose(1,2).reshape(B,C,H,W)

        # Gating
        gf  = self.gate(out.flatten(2).transpose(1,2))
        g1,g2 = gf.chunk(2,-1)
        gout  = self.gate_out(g1*g2)
        gout  = gout.transpose(1,2).reshape(B,C,H,W)

        return out + gout + x  # residual

# ── 4. Define LKDA-Fusion to replace RepC3 ─────────────────────
class LKDAFusionWrapper(nn.Module):
    """Replaces RepC3 fusion blocks with Large Kernel Dual-domain Attention"""
    def __init__(self, repc3, ch=256):
        super().__init__()
        self.repc3 = repc3

        # Large kernel branches (strip convolutions)
        self.lk1x1   = nn.Conv2d(ch,ch,1,padding=0,groups=ch,bias=False)
        self.lk1x15  = nn.Conv2d(ch,ch,(1,15),padding=(0,7),groups=ch,bias=False)
        self.lk15x1  = nn.Conv2d(ch,ch,(15,1),padding=(7,0),groups=ch,bias=False)
        self.lk_fuse = nn.Sequential(
            nn.Conv2d(ch,ch,1,bias=False),
            nn.BatchNorm2d(ch), nn.SiLU()
        )

        # Spatial attention
        self.sa_gap  = nn.AdaptiveAvgPool2d(1)
        self.sa_conv = nn.Conv2d(ch,ch,1)

        # Frequency attention
        self.fa_conv = nn.Conv2d(ch,ch,1)

    def forward(self, x):
        # Original RepC3 features
        base = self.repc3(x)

        # Large kernel attention
        lk = (self.lk1x1(base) +
              self.lk1x15(base) +
              self.lk15x1(base))
        lk = self.lk_fuse(lk)

        # Spatial attention (SCAM)
        sa = base * torch.sigmoid(self.sa_conv(self.sa_gap(base)))

        # Frequency attention (FCAM)
        xf  = torch.fft.rfft2(sa, norm="ortho")
        xf  = xf * torch.sigmoid(self.fa_conv(self.sa_gap(sa)))
        fa  = torch.fft.irfft2(xf, s=base.shape[-2:], norm="ortho")

        return base + lk + fa  # enhanced fusion

# ── 5. Apply all modifications ─────────────────────────────────
print("Applying ESO-DETR modifications...")

# Replace AIFI (model.11) with MMSA
old_aifi = torch_model[11]
torch_model[11] = MMSA_Module(dim=256, num_heads=8).to(device)
print("✅ AIFI → MMSA (model.11)")

# Wrap HGBlocks with GSHA (model.5, 6, 7 — deep backbone stages)
torch_model[5] = GHSAWrapper(torch_model[5], ch=1024).to(device)
torch_model[6] = GHSAWrapper(torch_model[6], ch=1024).to(device)
torch_model[7] = GHSAWrapper(torch_model[7], ch=1024).to(device)
print("✅ HGBlock → GSHA wrapper (model.5, 6, 7)")

# Replace RepC3 fusion blocks with LKDA-Fusion (model.16, 21)
torch_model[16] = LKDAFusionWrapper(torch_model[16], ch=256).to(device)
torch_model[21] = LKDAFusionWrapper(torch_model[21], ch=256).to(device)
print("✅ RepC3 → LKDA-Fusion (model.16, 21)")

# Count parameters
total = sum(p.numel() for p in torch_model.parameters())
print(f"\n✅ All ESO-DETR modifications applied!")
print(f"   Total parameters: {total/1e6:.2f}M")
print(f"   Paper target:     ~32M")

Applying ESO-DETR modifications...
✅ AIFI → MMSA (model.11)
✅ HGBlock → GSHA wrapper (model.5, 6, 7)
✅ RepC3 → LKDA-Fusion (model.16, 21)

✅ All ESO-DETR modifications applied!
   Total parameters: 46.02M
   Paper target:     ~32M


In [ ]:
#ESO EKLEME

!pip install ultralytics -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from ultralytics import RTDETR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ── 1. Load trained model ──────────────────────────────────────
model_rt = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/best.pt")
torch_model = model_rt.model.model

# ── 2. Define MMSA to replace AIFI ────────────────────────────
class MMSA_Module(nn.Module):
    """Replaces AIFI (model.11) with Multiscale Multihead Self-Attention"""
    def __init__(self, dim=256, num_heads=8):
        super().__init__()
        self.nh  = num_heads
        self.hd  = dim // num_heads #dim=kanal sayısı

        # Multiscale branch — 4 dilation rates
        self.ms_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(dim, dim//4, 1, bias=False),
                nn.Conv2d(dim//4, dim//4, 3, 1, d,
                          dilation=d, groups=dim//4, bias=False),
                nn.BatchNorm2d(dim//4), nn.SiLU()
            ) for d in [1, 2, 4, 8]
        ])
        self.ms_fuse = nn.Conv2d(dim, dim, 1, bias=False)

        # Channel attention branch
        self.ca_gap  = nn.AdaptiveAvgPool2d(1)
        self.ca_fc   = nn.Sequential(
            nn.Conv2d(dim, dim//16, 1), nn.ReLU(),
            nn.Conv2d(dim//16, dim, 1), nn.Sigmoid()
        )

        # Attention projections
        self.q  = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim*2)
        self.op = nn.Linear(dim, dim)

        # FFN
        self.n1  = nn.LayerNorm(dim)
        self.n2  = nn.LayerNorm(dim)
        self.ff  = nn.Sequential(
            nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))

    def forward(self, x, pos=None):
        B, C, H, W = x.shape

        # Multiscale features
        ms = self.ms_fuse(torch.cat([c(x) for c in self.ms_convs], 1))

        # Channel attention
        ca = x * self.ca_fc(self.ca_gap(x))

        # Flatten for attention
        q_in  = self.n1(x.flatten(2).transpose(1,2))
        kv_in = self.n1(ms.flatten(2).transpose(1,2))

        Q      = self.q(q_in)
        K, V   = self.kv(kv_in).chunk(2, -1)

        def split(t):
            return t.reshape(B,-1,self.nh,self.hd).transpose(1,2)

        Q,K,V  = split(Q), split(K), split(V)
        attn   = F.softmax(Q@K.transpose(-2,-1)/math.sqrt(self.hd), -1)
        out    = (attn@V).transpose(1,2).reshape(B,H*W,C)
        out    = self.op(out).transpose(1,2).reshape(B,C,H,W)

        # F5 = Attn + Channel_attn (Eq.8 in paper)
        f5     = out + ca

        # FFN
        f5f    = f5.flatten(2).transpose(1,2)
        f5f    = f5f + self.ff(self.n2(f5f))
        return f5f.transpose(1,2).reshape(B,C,H,W)

# ── 3. Define GSHA wrapper ─────────────────────────────────────
class GHSAWrapper(nn.Module):
    """Wraps existing HGBlock and adds GSHA attention on top"""
    def __init__(self, hgblock, ch):
        super().__init__()
        self.block = hgblock
        catt       = min(64, ch)

        # Single-head attention on partial channels
        self.ln   = nn.LayerNorm(catt)
        self.Wq   = nn.Linear(catt, catt, bias=False)
        self.Wk   = nn.Linear(catt, catt, bias=False)
        self.Wv   = nn.Linear(catt, catt, bias=False)
        self.Wo   = nn.Linear(ch, ch, bias=False)
        self.catt = catt

        # Gated unit
        self.gate = nn.Sequential(
            nn.LayerNorm(ch),
            nn.Linear(ch, ch*2),
            nn.SiLU()
        )
        self.gate_out = nn.Linear(ch, ch)

    def forward(self, x):
        x = self.block(x)
        B, C, H, W = x.shape

        # Split channels for SHA
        xatt = x[:,:self.catt].flatten(2).transpose(1,2)  # B,HW,catt
        xr   = x[:,self.catt:]                             # B,C-catt,H,W

        xn = self.ln(xatt)
        Q  = self.Wq(xn); K = self.Wk(xn); V = self.Wv(xn)
        a  = F.softmax(Q@K.transpose(-2,-1)/math.sqrt(self.catt), -1)
        xa = (a@V).transpose(1,2).reshape(B,self.catt,H,W)

        # Concat + project
        cat = torch.cat([xa,xr],1)
        out = self.Wo(cat.flatten(2).transpose(1,2))
        out = out.transpose(1,2).reshape(B,C,H,W)

        # Gating
        gf  = self.gate(out.flatten(2).transpose(1,2))
        g1,g2 = gf.chunk(2,-1)
        gout  = self.gate_out(g1*g2)
        gout  = gout.transpose(1,2).reshape(B,C,H,W)

        return out + gout + x  # residual

# ── 4. Define LKDA-Fusion to replace RepC3 ─────────────────────
class LKDAFusionWrapper(nn.Module):
    """Replaces RepC3 fusion blocks with Large Kernel Dual-domain Attention"""
    def __init__(self, repc3, ch=256):
        super().__init__()
        self.repc3 = repc3

        # Large kernel branches (strip convolutions)
        self.lk1x1   = nn.Conv2d(ch,ch,1,padding=0,groups=ch,bias=False)
        self.lk1x15  = nn.Conv2d(ch,ch,(1,15),padding=(0,7),groups=ch,bias=False)
        self.lk15x1  = nn.Conv2d(ch,ch,(15,1),padding=(7,0),groups=ch,bias=False)
        self.lk_fuse = nn.Sequential(
            nn.Conv2d(ch,ch,1,bias=False),
            nn.BatchNorm2d(ch), nn.SiLU()
        )

        # Spatial attention
        self.sa_gap  = nn.AdaptiveAvgPool2d(1)
        self.sa_conv = nn.Conv2d(ch,ch,1)

        # Frequency attention
        self.fa_conv = nn.Conv2d(ch,ch,1)

    def forward(self, x):
        # Original RepC3 features
        base = self.repc3(x)

        # Large kernel attention
        lk = (self.lk1x1(base) +
              self.lk1x15(base) +
              self.lk15x1(base))
        lk = self.lk_fuse(lk)

        # Spatial attention (SCAM)
        sa = base * torch.sigmoid(self.sa_conv(self.sa_gap(base)))

        # Frequency attention (FCAM)
        xf  = torch.fft.rfft2(sa, norm="ortho")
        xf  = xf * torch.sigmoid(self.fa_conv(self.sa_gap(sa)))
        fa  = torch.fft.irfft2(xf, s=base.shape[-2:], norm="ortho")

        return base + lk + fa  # enhanced fusion

# ── 5. Apply all modifications ─────────────────────────────────
print("Applying ESO-DETR modifications...")

# Replace AIFI (model.11) with MMSA
old_aifi = torch_model[11]
torch_model[11] = MMSA_Module(dim=256, num_heads=8).to(device)
print("✅ AIFI → MMSA (model.11)")

# Wrap HGBlocks with GSHA (model.5, 6, 7 — deep backbone stages)
torch_model[5] = GHSAWrapper(torch_model[5], ch=1024).to(device)
torch_model[6] = GHSAWrapper(torch_model[6], ch=1024).to(device)
torch_model[7] = GHSAWrapper(torch_model[7], ch=1024).to(device)
print("✅ HGBlock → GSHA wrapper (model.5, 6, 7)")

# Replace RepC3 fusion blocks with LKDA-Fusion (model.16, 21)
torch_model[16] = LKDAFusionWrapper(torch_model[16], ch=256).to(device)
torch_model[21] = LKDAFusionWrapper(torch_model[21], ch=256).to(device)
print("✅ RepC3 → LKDA-Fusion (model.16, 21)")

# Count parameters
total = sum(p.numel() for p in torch_model.parameters())
print(f"\n✅ All ESO-DETR modifications applied!")
print(f"   Total parameters: {total/1e6:.2f}M")
print(f"   Paper target:     ~32M")

Applying ESO-DETR modifications...
✅ AIFI → MMSA (model.11)
✅ HGBlock → GSHA wrapper (model.5, 6, 7)
✅ RepC3 → LKDA-Fusion (model.16, 21)

✅ All ESO-DETR modifications applied!
   Total parameters: 46.02M
   Paper target:     ~32M


In [ ]:
#ESO EKLEME

!pip install ultralytics -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from ultralytics import RTDETR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ── 1. Load trained model ──────────────────────────────────────
model_rt = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/best.pt")
torch_model = model_rt.model.model

# ── 2. Define MMSA to replace AIFI ────────────────────────────
class MMSA_Module(nn.Module):
    """Replaces AIFI (model.11) with Multiscale Multihead Self-Attention"""
    def __init__(self, dim=256, num_heads=8):
        super().__init__()
        self.nh  = num_heads
        self.hd  = dim // num_heads #dim=kanal sayısı

        # Multiscale branch — 4 dilation rates
        self.ms_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(dim, dim//4, 1, bias=False),
                nn.Conv2d(dim//4, dim//4, 3, 1, d,
                          dilation=d, groups=dim//4, bias=False),
                nn.BatchNorm2d(dim//4), nn.SiLU()
            ) for d in [1, 2, 4, 8]
        ])
        self.ms_fuse = nn.Conv2d(dim, dim, 1, bias=False)

        # Channel attention branch
        self.ca_gap  = nn.AdaptiveAvgPool2d(1)
        self.ca_fc   = nn.Sequential(
            nn.Conv2d(dim, dim//16, 1), nn.ReLU(),
            nn.Conv2d(dim//16, dim, 1), nn.Sigmoid()
        )

        # Attention projections
        self.q  = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim*2)
        self.op = nn.Linear(dim, dim)

        # FFN
        self.n1  = nn.LayerNorm(dim)
        self.n2  = nn.LayerNorm(dim)
        self.ff  = nn.Sequential(
            nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))

    def forward(self, x, pos=None):
        B, C, H, W = x.shape

        # Multiscale features
        ms = self.ms_fuse(torch.cat([c(x) for c in self.ms_convs], 1))

        # Channel attention
        ca = x * self.ca_fc(self.ca_gap(x))

        # Flatten for attention
        q_in  = self.n1(x.flatten(2).transpose(1,2))
        kv_in = self.n1(ms.flatten(2).transpose(1,2))

        Q      = self.q(q_in)
        K, V   = self.kv(kv_in).chunk(2, -1)

        def split(t):
            return t.reshape(B,-1,self.nh,self.hd).transpose(1,2)

        Q,K,V  = split(Q), split(K), split(V)
        attn   = F.softmax(Q@K.transpose(-2,-1)/math.sqrt(self.hd), -1)
        out    = (attn@V).transpose(1,2).reshape(B,H*W,C)
        out    = self.op(out).transpose(1,2).reshape(B,C,H,W)

        # F5 = Attn + Channel_attn (Eq.8 in paper)
        f5     = out + ca

        # FFN
        f5f    = f5.flatten(2).transpose(1,2)
        f5f    = f5f + self.ff(self.n2(f5f))
        return f5f.transpose(1,2).reshape(B,C,H,W)

# ── 3. Define GSHA wrapper ─────────────────────────────────────
class GHSAWrapper(nn.Module):
    """Wraps existing HGBlock and adds GSHA attention on top"""
    def __init__(self, hgblock, ch):
        super().__init__()
        self.block = hgblock
        catt       = min(64, ch)

        # Single-head attention on partial channels
        self.ln   = nn.LayerNorm(catt)
        self.Wq   = nn.Linear(catt, catt, bias=False)
        self.Wk   = nn.Linear(catt, catt, bias=False)
        self.Wv   = nn.Linear(catt, catt, bias=False)
        self.Wo   = nn.Linear(ch, ch, bias=False)
        self.catt = catt

        # Gated unit
        self.gate = nn.Sequential(
            nn.LayerNorm(ch),
            nn.Linear(ch, ch*2),
            nn.SiLU()
        )
        self.gate_out = nn.Linear(ch, ch)

    def forward(self, x):
        x = self.block(x)
        B, C, H, W = x.shape

        # Split channels for SHA
        xatt = x[:,:self.catt].flatten(2).transpose(1,2)  # B,HW,catt
        xr   = x[:,self.catt:]                             # B,C-catt,H,W

        xn = self.ln(xatt)
        Q  = self.Wq(xn); K = self.Wk(xn); V = self.Wv(xn)
        a  = F.softmax(Q@K.transpose(-2,-1)/math.sqrt(self.catt), -1)
        xa = (a@V).transpose(1,2).reshape(B,self.catt,H,W)

        # Concat + project
        cat = torch.cat([xa,xr],1)
        out = self.Wo(cat.flatten(2).transpose(1,2))
        out = out.transpose(1,2).reshape(B,C,H,W)

        # Gating
        gf  = self.gate(out.flatten(2).transpose(1,2))
        g1,g2 = gf.chunk(2,-1)
        gout  = self.gate_out(g1*g2)
        gout  = gout.transpose(1,2).reshape(B,C,H,W)

        return out + gout + x  # residual

# ── 4. Define LKDA-Fusion to replace RepC3 ─────────────────────
class LKDAFusionWrapper(nn.Module):
    """Replaces RepC3 fusion blocks with Large Kernel Dual-domain Attention"""
    def __init__(self, repc3, ch=256):
        super().__init__()
        self.repc3 = repc3

        # Large kernel branches (strip convolutions)
        self.lk1x1   = nn.Conv2d(ch,ch,1,padding=0,groups=ch,bias=False)
        self.lk1x15  = nn.Conv2d(ch,ch,(1,15),padding=(0,7),groups=ch,bias=False)
        self.lk15x1  = nn.Conv2d(ch,ch,(15,1),padding=(7,0),groups=ch,bias=False)
        self.lk_fuse = nn.Sequential(
            nn.Conv2d(ch,ch,1,bias=False),
            nn.BatchNorm2d(ch), nn.SiLU()
        )

        # Spatial attention
        self.sa_gap  = nn.AdaptiveAvgPool2d(1)
        self.sa_conv = nn.Conv2d(ch,ch,1)

        # Frequency attention
        self.fa_conv = nn.Conv2d(ch,ch,1)

    def forward(self, x):
        # Original RepC3 features
        base = self.repc3(x)

        # Large kernel attention
        lk = (self.lk1x1(base) +
              self.lk1x15(base) +
              self.lk15x1(base))
        lk = self.lk_fuse(lk)

        # Spatial attention (SCAM)
        sa = base * torch.sigmoid(self.sa_conv(self.sa_gap(base)))

        # Frequency attention (FCAM)
        xf  = torch.fft.rfft2(sa, norm="ortho")
        xf  = xf * torch.sigmoid(self.fa_conv(self.sa_gap(sa)))
        fa  = torch.fft.irfft2(xf, s=base.shape[-2:], norm="ortho")

        return base + lk + fa  # enhanced fusion

# ── 5. Apply all modifications ─────────────────────────────────
print("Applying ESO-DETR modifications...")

# Replace AIFI (model.11) with MMSA
old_aifi = torch_model[11]
torch_model[11] = MMSA_Module(dim=256, num_heads=8).to(device)
print("✅ AIFI → MMSA (model.11)")

# Wrap HGBlocks with GSHA (model.5, 6, 7 — deep backbone stages)
torch_model[5] = GHSAWrapper(torch_model[5], ch=1024).to(device)
torch_model[6] = GHSAWrapper(torch_model[6], ch=1024).to(device)
torch_model[7] = GHSAWrapper(torch_model[7], ch=1024).to(device)
print("✅ HGBlock → GSHA wrapper (model.5, 6, 7)")

# Replace RepC3 fusion blocks with LKDA-Fusion (model.16, 21)
torch_model[16] = LKDAFusionWrapper(torch_model[16], ch=256).to(device)
torch_model[21] = LKDAFusionWrapper(torch_model[21], ch=256).to(device)
print("✅ RepC3 → LKDA-Fusion (model.16, 21)")

# Count parameters
total = sum(p.numel() for p in torch_model.parameters())
print(f"\n✅ All ESO-DETR modifications applied!")
print(f"   Total parameters: {total/1e6:.2f}M")
print(f"   Paper target:     ~32M")

Applying ESO-DETR modifications...
✅ AIFI → MMSA (model.11)
✅ HGBlock → GSHA wrapper (model.5, 6, 7)
✅ RepC3 → LKDA-Fusion (model.16, 21)

✅ All ESO-DETR modifications applied!
   Total parameters: 46.02M
   Paper target:     ~32M


In [ ]:
# Fine-tune for 20 more epochs
model_rt.train(
    data="/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml",
    epochs=20,
    imgsz=640,
    batch=4,           # reduced for stability
    device=0,
    lr0=5e-5,          # low lr — we have good weights already
    optimizer="AdamW",
    project="/content/drive/MyDrive/rtdetr_visdrone",
    name="eso_detr",
    exist_ok=True,
    save=True,
    save_period=5,
    patience=10,
    freeze=10,         # freeze first 10 layers, only train modified parts
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=5e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/rtdetr_visdrone/rtdetr_fast/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=es

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/20      4.88G      2.175     0.1506     0.6671        571        640: 4% ╸─────────── 70/1618 3.0it/s 25.4s<8:44

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/20      5.91G      2.145     0.1501      0.667        259        640: 7% ╸─────────── 109/1618 3.2it/s 37.9s<7:58

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      19/20      11.5G     0.8896     0.4977     0.1004         52        640: 100% ━━━━━━━━━━━━ 1618/1618 3.9it/s 6:54
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.4it/s 8.2s
                   all        548      38759       0.42      0.307      0.252      0.138

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      20/20      2.44G      1.605     0.3105     0.1585        216        640: 0% ──────────── 0/1618  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      20/20      8.62G     0.8892      0.497    0.09868        103        640: 100% ━━━━━━━━━━━━ 1618/1618 3.9it/s 6:53
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.3it/s 8.3s
                   all        548      38759      0.416      0.307      0.253      0.139

20 epochs completed in 2.399 hours.
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/rtdetr_visdrone/eso_detr/weights/best.pt...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
rt-detr-l summary: 310 layers, 32,004,290 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 5.6it/s 12.3s
                   all        548      38759      0.409      0.3

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7cc992dc3b30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
from ultralytics import RTDETR

# Load last ESO checkpoint
model_eso = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/eso_detr/weights/last.pt")

# Continue for 30 more epochs
model_eso.train(
    data="/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml",
    epochs=30,
    imgsz=640,
    batch=4,
    device=0,
    lr0=2e-5,        # even lower lr now
    optimizer="AdamW",
    project="/content/drive/MyDrive/rtdetr_visdrone",
    name="eso_detr_continued",
    exist_ok=True,
    save=True,
    save_period=5,
    patience=15,
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=2e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/rtdetr_visdrone/eso_detr/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=eso

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30        11G      1.865     0.2524     0.3681         77        640: 100% ━━━━━━━━━━━━ 1618/1618 3.0it/s 9:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.4it/s 8.2s
                   all        548      38759     0.0467      0.109     0.0116    0.00225

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/30        11G      1.226     0.7142     0.1705        259        640: 0% ──────────── 0/1618  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30        11G      1.522      0.396     0.2178        345        640: 100% ━━━━━━━━━━━━ 1618/1618 3.1it/s 8:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.3it/s 8.3s
                   all        548      38759      0.162      0.226     0.0778     0.0371

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/30        11G     0.8818     0.5872    0.08922        221        640: 0% ──────────── 0/1618  0.3s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      13.6G     0.8545     0.4933     0.0923        137        640: 100% ━━━━━━━━━━━━ 1618/1618 3.2it/s 8:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.4it/s 8.3s
                   all        548      38759       0.46       0.35        0.3      0.168

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30      4.16G     0.8969     0.4694    0.06294        166        640: 0% ──────────── 0/1618  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      13.9G     0.8467     0.4939    0.08971         89        640: 100% ━━━━━━━━━━━━ 1618/1618 3.2it/s 8:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.2it/s 8.4s
                   all        548      38759      0.457      0.356      0.302      0.169

30 epochs completed in 4.326 hours.
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr_continued/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr_continued/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/rtdetr_visdrone/eso_detr_continued/weights/best.pt...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
rt-detr-l summary: 310 layers, 32,004,290 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 5.7it/s 12.1s
                   all        548 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7cc992d63200>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
from ultralytics import RTDETR

model_eso = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/eso_detr_continued/weights/last.pt")

model_eso.train(
    data="/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml",
    epochs=40,
    imgsz=640,
    batch=4,
    device=0,
    lr0=1e-4,
    lrf=0.01,
    optimizer="AdamW",
    momentum=0.9,
    weight_decay=1e-4,
    project="/content/drive/MyDrive/rtdetr_visdrone",
    name="eso_detr_paper_settings",
    exist_ok=True,
    save=True,
    save_period=5,
    patience=20,
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/rtdetr_visdrone/eso_detr_continued/weights/last.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0,

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/40      11.2G      1.765     0.2971     0.2915         77        640: 100% ━━━━━━━━━━━━ 1618/1618 2.9it/s 9:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.4it/s 8.2s
                   all        548      38759     0.0945     0.0853     0.0293    0.00715

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/40      4.48G      1.583     0.2975     0.2249        259        640: 0% ──────────── 0/1618  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/40      9.88G       1.36     0.4278     0.1641        345        640: 100% ━━━━━━━━━━━━ 1618/1618 3.1it/s 8:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.2it/s 8.4s
                   all        548      38759       0.32      0.268      0.194     0.0914

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/40      9.88G      0.924     0.4683    0.07899        221        640: 0% ──────────── 0/1618  0.3s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      39/40      11.8G     0.7606     0.4756     0.0706        188        640: 100% ━━━━━━━━━━━━ 1618/1618 3.2it/s 8:32
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.2it/s 8.4s
                   all        548      38759      0.548      0.417      0.384      0.221

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      40/40       4.3G     0.7164     0.4717    0.08814        128        640: 0% ──────────── 0/1618  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      40/40      10.5G     0.7539     0.4747    0.07069         85        640: 100% ━━━━━━━━━━━━ 1618/1618 3.2it/s 8:32
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.3it/s 8.3s
                   all        548      38759       0.55      0.416      0.386      0.222

40 epochs completed in 5.883 hours.
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr_paper_settings/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr_paper_settings/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/rtdetr_visdrone/eso_detr_paper_settings/weights/best.pt...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
rt-detr-l summary: 310 layers, 32,004,290 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 5.7it/s 12.1s
                   

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7cc629e280e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.4 MB/s eta 0:00:00


**ESO-DETR SON**

In [ ]:
from ultralytics import RTDETR

model_eso = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/eso_detr_paper_settings/weights/last.pt")

model_eso.train(
    data="/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml",
    epochs=30,
    imgsz=640,
    batch=4,
    device=0,
    lr0=1e-4,
    lrf=0.01,
    optimizer="AdamW",
    momentum=0.9,
    weight_decay=1e-4,
    project="/content/drive/MyDrive/rtdetr_visdrone",
    name="eso_detr_final30",
    exist_ok=True,
    save=True,
    save_period=5,
    patience=15,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hal

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      10.5G      1.666     0.3239     0.2303         77        640: 100% ━━━━━━━━━━━━ 1618/1618 3.2it/s 8:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 6.9it/s 10.1s
                   all        548      38759      0.348      0.177     0.0924     0.0225

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/30      10.5G      1.303     0.3862     0.1757        259        640: 0% ──────────── 0/1618  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      10.5G       1.19     0.4754     0.1228        345        640: 100% ━━━━━━━━━━━━ 1618/1618 3.4it/s 8:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.6it/s 8.0s
                   all        548      38759      0.388      0.279      0.214     0.0895

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/30      10.5G     0.9969      0.419    0.08175        221        640: 0% ──────────── 0/1618  0.3s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30        13G     0.7346     0.4701    0.06713        137        640: 100% ━━━━━━━━━━━━ 1618/1618 3.4it/s 7:57
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.5it/s 8.1s
                   all        548      38759      0.556      0.426      0.397      0.229

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30      3.59G     0.8249     0.4236    0.05029        166        640: 0% ──────────── 0/1618  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      13.4G     0.7265     0.4696     0.0658         89        640: 100% ━━━━━━━━━━━━ 1618/1618 3.4it/s 7:56
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 8.3it/s 8.3s
                   all        548      38759      0.566       0.42      0.395      0.229

30 epochs completed in 4.132 hours.
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr_final30/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/rtdetr_visdrone/eso_detr_final30/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/rtdetr_visdrone/eso_detr_final30/weights/best.pt...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
rt-detr-l summary: 310 layers, 32,004,290 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 5.8it/s 11.9s
                   all        548      3

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x791d440d6de0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

HIT-UAV

In [ ]:
import os

# 1. Kaggle API şifreni (resimdeki token) Colab'e tanıtıyoruz
os.environ['KAGGLE_API_TOKEN'] = "KGAT_aeaf8a5e6442d654801d94cd7be4029f"

# 2. Kaggle kütüphanesini kuralım
!pip install kaggle -q

# 3. HIT-UAV veri setini indiriyoruz
!kaggle datasets download -d pandrii000/hituav-a-highaltitude-infrared-thermal-dataset

# 4. Zip'ten 'hit_uav_veri_seti' klasörüne çıkarıyoruz
!unzip -q hituav-a-highaltitude-infrared-thermal-dataset.zip -d hit_uav_veri_seti

# 5. İşi biten Zip dosyasını silip yer açıyoruz
!rm hituav-a-highaltitude-infrared-thermal-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/pandrii000/hituav-a-highaltitude-infrared-thermal-dataset
License(s): GNU Lesser General Public License 3.0
100% 191M/191M [00:10<00:00, 18.7MB/s]



In [ ]:
import os
train = len(os.listdir("/content/drive/MyDrive/hit_uav_veri_seti/hit-uav/images/train"))
val   = len(os.listdir("/content/drive/MyDrive/hit_uav_veri_seti/hit-uav/images/val"))
test  = len(os.listdir("/content/drive/MyDrive/hit_uav_veri_seti/hit-uav/images/test"))
print(f"Train: {train}")
print(f"Val:   {val}")
print(f"Test:  {test}")

Train: 2008
Val:   287
Test:  571


In [ ]:
# Copy from Drive to local Colab storage
!cp -r "/content/drive/MyDrive/hit_uav_veri_seti/hit-uav" /content/hit-uav
print("✅ Copied to local storage")

# Verify
!echo "Train images:" && ls /content/hit-uav/images/train | wc -l
!echo "Val images:"   && ls /content/hit-uav/images/val   | wc -l

✅ Copied to local storage
Train images:
2008
Val images:
287


**ESO-DETR HIT-UAV**

In [ ]:
from ultralytics import RTDETR

model_eso = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/eso_detr_final30/weights/best.pt")

model_eso.train(
    data="/content/hit-uav/dataset.yaml",
    epochs=30,
    imgsz=640,
    batch=4,
    device=0,
    lr0=1e-4,
    lrf=0.01,
    optimizer="AdamW",
    momentum=0.9,
    weight_decay=1e-4,
    project="/content/drive/MyDrive/rtdetr_hituav",
    name="eso_detr_hituav",
    exist_ok=True,
    save=True,
    save_period=5,
    patience=15,
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hit-uav/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/rtdetr_visdrone/eso_detr_final30/weights/best.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=eso_detr_hituav, nbs=64, nms=False, opset=Non

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      3.64G       1.27     0.7344     0.2235         22        640: 100% ━━━━━━━━━━━━ 502/502 3.1it/s 2:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 5.0it/s 7.2s
                   all        287       2460      0.236      0.174     0.0346     0.0121

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/30      3.83G      1.112     0.5497     0.1301         68        640: 0% ──────────── 0/502  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      3.84G     0.8675     0.8097     0.1146         40        640: 100% ━━━━━━━━━━━━ 502/502 3.4it/s 2:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 9.8it/s 3.7s
                   all        287       2460      0.286      0.307     0.0882     0.0383

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/30      3.84G     0.8845     0.7063    0.08346         48        640: 0% ──────────── 0/502  0.3s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      3.84G     0.4589      0.444    0.06612         12        640: 100% ━━━━━━━━━━━━ 502/502 3.4it/s 2:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 9.8it/s 3.7s
                   all        287       2460      0.838      0.585      0.646      0.392

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30      3.84G     0.4521     0.4358    0.05161         31        640: 0% ──────────── 0/502  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      3.84G     0.4489     0.4399    0.06399         59        640: 100% ━━━━━━━━━━━━ 502/502 3.4it/s 2:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 9.8it/s 3.7s
                   all        287       2460      0.874      0.607      0.686      0.424

30 epochs completed in 1.287 hours.
Optimizer stripped from /content/drive/MyDrive/rtdetr_hituav/eso_detr_hituav/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/rtdetr_hituav/eso_detr_hituav/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/rtdetr_hituav/eso_detr_hituav/weights/best.pt...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
rt-detr-l summary: 310 layers, 31,994,015 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 9.4it/s 3.8s
                   all        287       2460      0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d694c4e9310>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [ ]:
from ultralytics import RTDETR

model_eso = RTDETR(
    "/content/drive/MyDrive/rtdetr_visdrone/eso_detr_final30/weights/best.pt")

model_eso.train(
    data="/content/hit-uav/dataset.yaml",
    epochs=30,
    imgsz=640,
    batch=4,
    device=0,
    lr0=1e-4,
    lrf=0.01,
    optimizer="AdamW",
    momentum=0.9,
    weight_decay=1e-4,
    project="/content/drive/MyDrive/rtdetr_hituav",
    name="eso_detr_hituav_4class",
    exist_ok=True,
    save=True,
    save_period=5,
    patience=15,
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hit-uav/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/rtdetr_visdrone/eso_detr_final30/weights/best.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=eso_detr_hituav_4class, nbs=64, nms=False, op

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30       4.1G       2.34      4.005     0.6145         19        640: 1% ──────────── 3/502 3.2it/s 1.3s<2:34


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
%%javascript
function KeepAlive() {
  document.querySelector("#top-toolbar").click();
}
setInterval(KeepAlive, 60000);

<IPython.core.display.Javascript object>

# **# yolov26 da da test et**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Create a new YOLO model from scratch
model = YOLO("yolo26n.yaml")

# Load a pretrained YOLO model (recommended for training)
model = YOLO("yolo26n.pt")

# Train the model using the 'coco8.yaml' dataset for 3 epochs
results = model.train(data="coco8.yaml", epochs=3)

# Evaluate the model's performance on the validation set
results = model.val()

# Perform object detection on an image using the model
results = model("https://ultralytics.com/images/bus.jpg")

# Export the model to ONNX format
success = model.export(format="onnx")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=Fals

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.93...
ONNX: export success ✅ 7.0s, saved as '/content/runs/detect/train/weights/best.onnx' (9.5 MB)

Export complete (7.2s)
Results saved to /content/runs/detect/train/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/train/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/train/weights/best.onnx imgsz=640 data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/coco8.yaml  
Visualize:       https://netron.app


In [ ]:
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# VisDrone2019-DET dataset https://github.com/VisDrone/VisDrone-Dataset by Tianjin University
# Documentation: https://docs.ultralytics.com/datasets/detect/visdrone/
# Example usage: yolo train data=VisDrone.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── VisDrone ← downloads here (2.3 GB)
%%writefile VisDrone.yaml
# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: VisDrone # dataset root dir
train: images/train # train images (relative to 'path') 6471 images
val: images/val # val images (relative to 'path') 548 images
test: images/test # test-dev images (optional) 1610 images

# Classes
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor

# Download script/URL (optional) ---------------------------------------------------------------------------------------
download : |
  import os
  from pathlib import Path
  import shutil

  from ultralytics.utils.downloads import download
  from ultralytics.utils import ASSETS_URL, TQDM

  def visdrone2yolo(dir, split, source_name=None):
      """Convert VisDrone annotations to YOLO format with images/{split} and labels/{split} structure."""
      from PIL import Image

      source_dir = dir / (source_name or f"VisDrone2019-DET-{split}")
      images_dir = dir / "images" / split
      labels_dir = dir / "labels" / split
      labels_dir.mkdir(parents=True, exist_ok=True)

      # Move images to new structure
      if (source_images_dir := source_dir / "images").exists():
          images_dir.mkdir(parents=True, exist_ok=True)
          for img in source_images_dir.glob("*.jpg"):
              img.rename(images_dir / img.name)

      for f in TQDM((source_dir / "annotations").glob("*.txt"), desc=f"Converting {split}"):
          img_size = Image.open(images_dir / f.with_suffix(".jpg").name).size
          dw, dh = 1.0 / img_size[0], 1.0 / img_size[1]
          lines = []

          with open(f, encoding="utf-8") as file:
              for row in [x.split(",") for x in file.read().strip().splitlines()]:
                  if row[4] != "0":  # Skip ignored regions
                      x, y, w, h = map(int, row[:4])
                      cls = int(row[5]) - 1
                      # Convert to YOLO format
                      x_center, y_center = (x + w / 2) * dw, (y + h / 2) * dh
                      w_norm, h_norm = w * dw, h * dh
                      lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

          (labels_dir / f.name).write_text("".join(lines), encoding="utf-8")

  # Download (ignores test-challenge split)
  dir = Path(yaml["path"])  # dataset root dir
  urls = [
      f"{ASSETS_URL}/VisDrone2019-DET-train.zip",
      f"{ASSETS_URL}/VisDrone2019-DET-val.zip",
      f"{ASSETS_URL}/VisDrone2019-DET-test-dev.zip",
      # f"{ASSETS_URL}/VisDrone2019-DET-test-challenge.zip",
  ]
  download(urls, dir=dir, threads=4)

  # Convert
  splits = {"VisDrone2019-DET-train": "train", "VisDrone2019-DET-val": "val", "VisDrone2019-DET-test-dev": "test"}
  for folder, split in splits.items():
      visdrone2yolo(dir, split, folder)  # convert VisDrone annotations to YOLO labels
      shutil.rmtree(dir / folder)  # cleanup original directory

Writing VisDrone.yaml


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="VisDrone.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    cache=True,
    workers=4,
    device=0
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0,

KeyboardInterrupt: 

In [ ]:
!pip install ultralytics -q --upgrade
import ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from ultralytics import YOLO

# En büyük model
model = YOLO("yolo26x.pt")  # extra-large

results = model.train(
    data="VisDrone.yaml",
    epochs=150,              # daha fazla epoch
    imgsz=1280,              # daha yüksek çözünürlük ← küçük nesneler için kritik!
    batch=4,                 # imgsz=1280 için batch küçültüldü
    device=0,
    lr0=5e-4,
    lrf=0.01,
    optimizer="AdamW",
    momentum=0.9,
    weight_decay=1e-4,
    # Augmentation
    mosaic=1.0,
    close_mosaic=15,
    scale=0.5,
    fliplr=0.5,
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    # Kaydetme
    save=True,
    save_period=5,
    patience=30,
    project="/content/drive/MyDrive/yolo26_visdrone",
    name="yolo26x_best",
    exist_ok=True,
    cache=True,
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26x.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=yolo26x_best, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, perspective

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

# Kaldığı yerden devam et
model = YOLO("/content/drive/MyDrive/yolo26_visdrone/yolo26x_best/weights/last.pt")

model.train(
    data="VisDrone.yaml",
    epochs=150,
    imgsz=1280,
    batch=2,           # ← 4'ten 2'ye düşürüldü
    device=0,
    lr0=5e-4,
    lrf=0.01,
    optimizer="AdamW",
    momentum=0.9,
    weight_decay=1e-4,
    mosaic=1.0,
    close_mosaic=15,
    save=True,
    save_period=5,
    patience=30,
    project="/content/drive/MyDrive/yolo26_visdrone",
    name="yolo26x_best",
    resume=True,       # ← kaldığı yerden devam
    exist_ok=True,
    cache=True,
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/yolo26_visdrone/yolo26x_best/weights/last.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=yolo26x_best, nbs=64, nms=False, opset=None, optimize=False, opt

In [ ]:
#TINY PERSON DATASET
# 1. gdown kütüphanesini kullanarak dosyayı doğrudan Colab'e indiriyoruz
!gdown --id "1nWo9mRymsGRbmJQcMAnVnC-aWK7PtWby" -O /content/test.tar.gz

print("Paket açılıyor, lütfen bekle Sude...")
!tar -xf /content/test.tar.gz -C /content/

import os
print("İçerik listeleniyor:")
print(os.listdir('/content/'))

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1nWo9mRymsGRbmJQcMAnVnC-aWK7PtWby
From (redirected): https://drive.google.com/uc?id=1nWo9mRymsGRbmJQcMAnVnC-aWK7PtWby&confirm=t&uuid=d56495c2-f426-4395-a1e2-376a2b634de0
To: /content/test.tar.gz
100% 574M/574M [00:12<00:00, 44.2MB/s]
Paket açılıyor, lütfen bekle Sude...
İçerik listeleniyor:
['.config', 'datasets', 'test.tar.gz', 'drive', 'test', 'sample_data']


**YOLOV26 TRAIN**

In [ ]:
#train

from ultralytics import YOLO

# Load a model
model = YOLO("yolo26x.yaml")  # build a new model from YAML
model = YOLO("yolo26x.pt")  # load a pretrained model (recommended for training)
model = YOLO("yolo26x.yaml").load("yolo26x.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data="VisDrone.yaml", epochs=150, imgsz=640,batch=16)

Transferred 1092/1092 items from pretrained weights
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26x.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimiz

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-2/weights/last.pt")

model.train(resume=True)

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train-2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nm

YOLOV26 VSDRONE19 VAL

In [ ]:
#val
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n.pt")  # load an official model
model = YOLO("/content/runs/detect/train/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category
metrics.box.image_metrics  # per-image metrics dictionary with precision, recall, F1, TP, FP, and FN

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2525.1±1158.0 MB/s, size: 126.7 KB)
val: Scanning /content/datasets/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 121.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 4.4it/s 7.9s
                   all        548      38759       0.45      0.341      0.317      0.176
            pedestrian        520       8844      0.473      0.376      0.354      0.144
                people        482       5125      0.477      0.277      0.266     0.0919
               bicycle        364       1287      0.253      0.111     0.0833     0.0304
                   car        515      14064      0.634      0.757      0.738      0.484
                   van        421  

{'0000360_05685_d_0000742.jpg': {'precision': 0.15666666666666668,
  'recall': 0.7833333333333333,
  'f1': 0.2611111111111111,
  'tp': 47,
  'fp': 253,
  'fn': 13},
 '0000360_06273_d_0000745.jpg': {'precision': 0.05,
  'recall': 0.8823529411764706,
  'f1': 0.09463722397476342,
  'tp': 15,
  'fp': 285,
  'fn': 2},
 '0000360_06665_d_0000747.jpg': {'precision': 0.14666666666666667,
  'recall': 0.9166666666666666,
  'f1': 0.25287356321839083,
  'tp': 44,
  'fp': 256,
  'fn': 4},
 '0000360_06861_d_0000748.jpg': {'precision': 0.08333333333333333,
  'recall': 1.0,
  'f1': 0.15384615384615385,
  'tp': 25,
  'fp': 275,
  'fn': 0},
 '0000360_07057_d_0000749.jpg': {'precision': 0.05,
  'recall': 1.0,
  'f1': 0.09523809523809523,
  'tp': 15,
  'fp': 285,
  'fn': 0},
 '0000360_07253_d_0000750.jpg': {'precision': 0.09,
  'recall': 0.9642857142857143,
  'f1': 0.16463414634146342,
  'tp': 27,
  'fp': 273,
  'fn': 1},
 '0000360_07645_d_0000752.jpg': {'precision': 0.06037735849056604,
  'recall': 1.0,
 

HIT-UAV TRAIN

In [ ]:
from ultralytics import YOLO

# Create dataset.yaml for HIT-UAV
with open("/content/drive/MyDrive/yolov26/datasets/hit-uav/dataset.yaml", "w") as f:
    f.write("path: /content/drive/MyDrive/yolov26/datasets/hit-uav\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("nc: 5\n")
    f.write("names: ['Person', 'Car', 'Bicycle', 'OtherVehicle', 'DontCare']\n")

# Load a model
model = YOLO("yolo26n.yaml")  # build a new model from YAML
model = YOLO("yolo26n.pt")  # load a pretrained model (recommended for training)
model = YOLO("yolo26n.yaml").load("yolo26n.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data="/content/drive/MyDrive/yolov26/datasets/hit-uav/dataset.yaml", epochs=150, imgsz=640)

Transferred 708/708 items from pretrained weights
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolov26/datasets/hit-uav/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, n

HIT-UAV VAL

In [ ]:
#val
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n.pt")  # load an official model
model = YOLO("/content/runs/detect/train-2/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category
metrics.box.image_metrics  # per-image metrics dictionary with precision, recall, F1, TP, FP, and FN

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,811 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 47.6±12.1 MB/s, size: 65.9 KB)
val: Scanning /content/drive/MyDrive/yolov26/datasets/hit-uav/labels/val.cache... 287 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 287/287 86.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 4.7it/s 3.9s
                   all        287       2460      0.827      0.765      0.819      0.519
                Person        171       1168      0.901      0.881      0.921      0.503
                   Car        136        719      0.926      0.957      0.982      0.736
               Bicycle         53        554       0.89      0.834       0.91      0.525
          OtherVehicle          9         12      0.813      0.726      0.781      0.645
              DontCa

{'1_70_70_0_07716.jpg': {'precision': 0.3194444444444444,
  'recall': 1.0,
  'f1': 0.4842105263157894,
  'tp': 23,
  'fp': 49,
  'fn': 0},
 '1_70_70_0_07719.jpg': {'precision': 0.75,
  'recall': 1.0,
  'f1': 0.8571428571428571,
  'tp': 6,
  'fp': 2,
  'fn': 0},
 '1_70_80_0_07736.jpg': {'precision': 0.35714285714285715,
  'recall': 0.9615384615384616,
  'f1': 0.5208333333333334,
  'tp': 25,
  'fp': 45,
  'fn': 1},
 '1_70_80_0_07741.jpg': {'precision': 0.325,
  'recall': 1.0,
  'f1': 0.4905660377358491,
  'tp': 26,
  'fp': 54,
  'fn': 0},
 '1_70_80_0_07751.jpg': {'precision': 0.27586206896551724,
  'recall': 1.0,
  'f1': 0.4324324324324324,
  'tp': 8,
  'fp': 21,
  'fn': 0},
 '1_70_90_0_07786.jpg': {'precision': 0.35714285714285715,
  'recall': 1.0,
  'f1': 0.5263157894736842,
  'tp': 10,
  'fp': 18,
  'fn': 0},
 '1_70_90_0_07793.jpg': {'precision': 0.45,
  'recall': 0.9473684210526315,
  'f1': 0.6101694915254237,
  'tp': 18,
  'fp': 22,
  'fn': 1},
 '1_70_90_0_07797.jpg': {'precision': 

vsdrone->eso=39 yolov26=31
hıt-uav->eso=68 yolov26=81

In [ ]:
!unzip -q /content/datasets/hituav_dataset.zip -d /content/datasets/

FASTER R-CNN

In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu128
0.26.0+cu128


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn

model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
print("Model yüklendi.")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 209MB/s]


Model yüklendi.


In [ ]:
import zipfile

with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/VisDrone2019-DET-train.zip', 'r') as z:
    z.extractall('/content/VisDrone')

with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/VisDrone2019-DET-val.zip', 'r') as z:
    z.extractall('/content/VisDrone')

In [ ]:
import os

print(os.listdir('/content/VisDrone'))

['VisDrone2019-DET-train', 'VisDrone2019-DET-val']


In [ ]:
#annotations formatı
ann_dir = "/content/VisDrone/VisDrone2019-DET-train/annotations"

sample = os.listdir(ann_dir)[0]

with open(f"{ann_dir}/{sample}") as f:
    print(f.read())

837,890,247,160,1,5,1,0
842,726,111,155,1,4,0,0
838,627,102,104,1,4,0,0
477,611,190,93,1,4,0,1
858,484,60,68,1,4,0,0
1097,599,157,71,1,7,0,1
347,439,68,44,1,4,0,1
427,427,75,42,1,4,0,0
511,460,77,41,1,4,0,2
462,467,99,50,1,4,0,2
383,494,150,60,1,4,0,2
317,541,101,47,1,4,0,2
223,558,160,75,1,4,0,2
107,636,191,79,1,4,0,2
591,414,83,37,1,5,0,1
720,417,89,38,1,4,0,0
816,392,33,29,1,4,0,0
857,369,33,32,1,4,0,0
645,461,115,40,1,4,0,1
595,498,138,55,1,4,0,2
564,526,137,63,1,4,0,2
516,550,171,51,1,4,0,2
488,582,182,68,1,4,0,2
1196,594,43,69,1,1,0,1
927,455,115,32,1,4,0,1
931,437,84,37,1,4,0,1
915,425,96,32,1,4,0,2
911,410,88,39,1,4,0,2
909,401,88,36,1,4,0,2



In [ ]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset
class VisDroneDataset(Dataset):

    def __init__(self, root):
        self.root = root

        self.img_dir = os.path.join(root, "images")
        self.ann_dir = os.path.join(root, "annotations")

        self.images = sorted(os.listdir(self.img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)

        ann_path = os.path.join(
            self.ann_dir,
            img_name.replace(".jpg", ".txt")
        )

        image = Image.open(img_path).convert("RGB")

        boxes = []
        labels = []

        with open(ann_path) as f:

            for line in f:

                row = line.strip().split(",")

                x = float(row[0])
                y = float(row[1])
                w = float(row[2])
                h = float(row[3])

                cls = int(row[5])

                if cls == 0:
                    continue

                xmin = x
                ymin = y
                xmax = x + w
                ymax = y + h

                boxes.append([xmin, ymin, xmax, ymax])

                labels.append(cls)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)

        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        image = torchvision.transforms.ToTensor()(image)

        return image, target

In [ ]:
train_dataset = VisDroneDataset(
    "/content/VisDrone/VisDrone2019-DET-train"
)

In [ ]:
val_dataset = VisDroneDataset(
    "/content/VisDrone/VisDrone2019-DET-val"
)

In [ ]:
img, target = train_dataset[0]

print(img.shape)
print(target)

torch.Size([3, 540, 960])
{'boxes': tensor([[708., 471., 782., 504.],
        [639., 425., 700., 471.],
        [594., 399., 658., 450.],
        [562., 390., 623., 428.],
        [540., 372., 605., 405.],
        [514., 333., 582., 368.],
        [501., 317., 565., 348.],
        [501., 299., 546., 327.],
        [489., 284., 537., 311.],
        [463., 262., 511., 291.],
        [458., 252., 507., 274.],
        [448., 242., 493., 262.],
        [442., 230., 491., 249.],
        [439., 214., 484., 235.],
        [429., 208., 471., 227.],
        [420., 199., 463., 219.],
        [398., 188., 439., 206.],
        [ 46., 391.,  60., 417.],
        [421., 433., 495., 477.],
        [369., 346., 433., 380.],
        [398., 410., 470., 456.],
        [394., 393., 464., 429.],
        [377., 364., 448., 402.],
        [357., 312., 415., 343.],
        [359., 298., 413., 320.],
        [348., 283., 391., 311.],
        [345., 271., 397., 290.],
        [340., 260., 400., 278.],
        [340

In [ ]:
#faster r-cnn için dataloader oluşturmak gerekiyo
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

In [ ]:
images, targets = next(iter(train_loader))

print(len(images))
print(targets[0].keys())

2
dict_keys(['boxes', 'labels'])


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn

model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

num_classes = 12  # 10 sınıf + background

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes
)

In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

In [ ]:
for epoch in range(150):

    model.train()

    for images, targets in train_loader:

        loss_dict = model(images, targets)

        loss = sum(loss_dict.values())

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    print(epoch, loss.item())

KeyboardInterrupt: 

In [ ]:
#DENEME
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import os
from PIL import Image
from torch.utils.data import DataLoader
import torchvision
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ═══════════════════════════════════════════════════════════════
# 1. Eğitilmiş modeli yükle
# ═══════════════════════════════════════════════════════════════
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
num_classes = 11
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# En iyi checkpoint'i yükle
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/faster_rcnn_epoch_50.pth"
))
model = model.to(device)
model.eval()

print("Model yüklendi!")

# ═══════════════════════════════════════════════════════════════
# 2. Val dataset üzerinde tahmin yap
# ═══════════════════════════════════════════════════════════════
val_dir = "/content/VisDrone/VisDrone2019-DET-val"
val_images = sorted(os.listdir(os.path.join(val_dir, "images")))

# COCO format predictions
coco_predictions = []
image_ids = {}

for img_idx, img_name in enumerate(val_images):
    img_path = os.path.join(val_dir, "images", img_name)
    image = Image.open(img_path).convert("RGB")
    image_tensor = torchvision.transforms.ToTensor()(image).to(device)

    # Tahmin yap
    with torch.no_grad():
        predictions = model([image_tensor])

    # COCO format'a çevir
    boxes = predictions[0]["boxes"].cpu().numpy()
    scores = predictions[0]["scores"].cpu().numpy()
    labels = predictions[0]["labels"].cpu().numpy()

    # Confidence threshold
    confidence_threshold = 0.5
    valid_indices = scores >= confidence_threshold

    for i in np.where(valid_indices)[0]:
        x1, y1, x2, y2 = boxes[i]
        w = x2 - x1
        h = y2 - y1

        coco_predictions.append({
            "image_id": img_idx,
            "category_id": int(labels[i]),
            "bbox": [float(x1), float(y1), float(w), float(h)],
            "score": float(scores[i])
        })

print(f"Total predictions: {len(coco_predictions)}")

# ═══════════════════════════════════════════════════════════════
# 3. Ground truth oluştur (COCO format)
# ═══════════════════════════════════════════════════════════════
coco_annotations = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": i, "name": f"class_{i}"} for i in range(1, 11)
    ]
}

annotation_id = 0
for img_idx, img_name in enumerate(val_images):
    img_path = os.path.join(val_dir, "images", img_name)
    image = Image.open(img_path)

    coco_annotations["images"].append({
        "id": img_idx,
        "file_name": img_name,
        "height": image.height,
        "width": image.width
    })

    # Annotations
    ann_path = os.path.join(val_dir, "annotations", img_name.replace(".jpg", ".txt"))
    with open(ann_path) as f:
        for line in f:
            row = line.strip().split(",")
            if len(row) < 6:
                continue

            x = float(row[0])
            y = float(row[1])
            w = float(row[2])
            h = float(row[3])
            cls = int(row[5])

            if cls == 0:
                continue

            coco_annotations["annotations"].append({
                "id": annotation_id,
                "image_id": img_idx,
                "category_id": cls,
                "bbox": [x, y, w, h],
                "area": w * h,
                "iscrowd": 0
            })
            annotation_id += 1

# COCO GT JSON kaydet
with open("/tmp/coco_gt.json", "w") as f:
    json.dump(coco_annotations, f)

# Predictions JSON kaydet
with open("/tmp/coco_predictions.json", "w") as f:
    json.dump(coco_predictions, f)

print(f"GT annotations: {len(coco_annotations['annotations'])}")

# ═══════════════════════════════════════════════════════════════
# 4. mAP Hesapla
# ═══════════════════════════════════════════════════════════════
coco_gt = COCO("/tmp/coco_gt.json")
coco_predictions_api = coco_gt.loadRes("/tmp/coco_predictions.json")

coco_eval = COCOeval(coco_gt, coco_predictions_api, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

print("\n" + "="*50)
print("FASTER R-CNN SONUÇLARI")
print("="*50)
print(f"mAP@0.5:0.95 : {coco_eval.stats[0]:.4f}")
print(f"mAP@0.5      : {coco_eval.stats[1]:.4f}")
print(f"mAP@0.75     : {coco_eval.stats[2]:.4f}")
print("="*50)

In [ ]:
#DENEME
import os
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, Dataset
from PIL import Image

# ═══════════════════════════════════════════════════════════════
# 1. VERİ OKUYUCU (CUSTOM DATASET)
# ═══════════════════════════════════════════════════════════════
class VisDroneDataset(Dataset):
    def __init__(self, root):
        self.root = root
        self.imgs = list(sorted(os.listdir(os.path.join(root, "images"))))

    def __getitem__(self, idx):
        # Resim ve etiket yollarını bul
        img_name = self.imgs[idx]
        img_path = os.path.join(self.root, "images", img_name)
        ann_path = os.path.join(self.root, "annotations", img_name.replace(".jpg", ".txt"))

        # Resmi yükle
        img = Image.open(img_path).convert("RGB")
        img_tensor = torchvision.transforms.ToTensor()(img)

        boxes = []
        labels = []

        # Etiket dosyasını (TXT) oku
        if os.path.exists(ann_path):
            with open(ann_path) as f:
                for line in f:
                    row = line.strip().split(",")
                    if len(row) < 6:
                        continue

                    x, y, w, h = float(row[0]), float(row[1]), float(row[2]), float(row[3])
                    cls = int(row[5])

                    # 0 (Göz ardı edilenler) ve 10'dan büyük (11 - Diğerleri) sınıfları atla
                    if cls == 0 or cls > 10:
                        continue

                    # Faster R-CNN kutuları [x1, y1, x2, y2] formatında ister
                    boxes.append([x, y, x + w, y + h])
                    labels.append(cls)

        # Eğer resimde hiç nesne yoksa boş tensör oluştur
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        # PyTorch'un beklediği target sözlüğü
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])

        return img_tensor, target

    def __len__(self):
        return len(self.imgs)

# Farklı boyuttaki resimleri aynı batch'e koyabilmek için gerekli fonksiyon
def collate_fn(batch):
    return tuple(zip(*batch))

# ═══════════════════════════════════════════════════════════════
# 2. MODEL VE VERİ YÜKLEYİCİ (DATALOADER) AYARLARI
# ═══════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SENİN KLASÖR YOLUN (Burayı kontrol et)
train_dir = "/content/VisDrone/VisDrone2019-DET-train"

# Veri setini oluştur
train_dataset = VisDroneDataset(train_dir)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

# Faster R-CNN ResNet-50 omurgasını çek
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

# Sınıf sayısını güncelle (VisDrone'da 10 sınıf var + 1 Arka Plan = 11)
num_classes = 11
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(device)

# Optimizasyon ayarları
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# ═══════════════════════════════════════════════════════════════
# 3. EĞİTİM (TRAINING) DÖNGÜSÜ
# ═══════════════════════════════════════════════════════════════
# EPOCH SAYISINI BURADAN DEĞİŞTİREBİLİRSİN
num_epochs = 10

print(f"Eğitim başlıyor... Kullanılan cihaz: {device}")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for i, (images, targets) in enumerate(train_loader):
        # Resimleri ve etiketleri ekran kartına (GPU) yolla
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Modeli çalıştır ve kayıp (loss) değerlerini al
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Geriye yayılım (Backpropagation) ve ağırlık güncelleme
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

        # Her 100 adımda bir ekrana bilgi yazdır
        if i % 100 == 0:
            print(f"Epoch: [{epoch+1}/{num_epochs}], Adım: [{i}/{len(train_loader)}], Loss: {losses.item():.4f}")

    print(f"--- Epoch {epoch+1} Bitti! Ortalama Loss: {epoch_loss/len(train_loader):.4f} ---")

    # Modeli Drive'a (veya Colab klasörüne) kaydet
    save_path = f"/content/faster_rcnn_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Model kaydedildi: {save_path}\n")

print("Eğitim başarıyla tamamlandı! 🎉")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
